Binned data (e.g.:
-Geburtsland (Gruppen): Insgesamt_Bevoelkerung_Geburtsland_Gruppen_10km-Gitter + Germany/EU27/… bins.
-Staatsangehörigkeit (Gruppen): Insgesamt_Bevoelkerung_Staatsangehoerigkeit_Gruppen_10km-Gitter + Germany/EU27/… bins.
-Wohnungsfläche (10 m²-Intervalle): Insgesamt_Wohnungen_Flaeche_der_Wohnung_10m2_Intervalle_* + unter30…180+.
-Wohnungen nach Zimmerzahl: Insgesamt_Wohnungen_Wohnungen_nach_Zahl_der_Raeume_* + 1 Raum…7+.
etc.)
Processed similarly to ages

In [1]:
# -*- coding: utf-8 -*-
"""
Generic hierarchical downscaling for NON-AGE categorical vectors.

Core idea (per topic):
  - Child HARD rows  : child topic total column (e.g., "Insgesamt_*") [optionally *_adj]
  - Parent HARD cols : parent per-category totals (sum over categories at parent level)
  - Trust-blended local -> parent shares (configurable w_min / t_pc) + damping alpha
  - Robust raking/IPF to satisfy both margins

Configure topics in build_topic_specs_for_level(), run 10km→1km and/or 1km→100m.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from tqdm import tqdm

# for memory-safe full-output writing at 100m
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# ---------------------------------------------------------------------
# 0) Trust blending (central, easy-to-tune)
# ---------------------------------------------------------------------

@dataclass
class TrustBlend:
    """
    Compute local trust weight w_i in [w_min, 1] from child row totals T_i.
    - w_min : baseline trust even for tiny totals
    - t_pc  : "people per category" target; when T_i ≈ K * t_pc, w_i ~ 1
              (K = number of categories in the topic)
    """
    w_min: float = 0.40
    t_pc: float  = 5.0

    def weights(self, totals: np.ndarray, n_categories: int) -> np.ndarray:
        cutoff = max(self.t_pc * max(n_categories, 1), 1e-9)
        base = np.minimum(1.0, totals / cutoff)
        return self.w_min + (1.0 - self.w_min) * base


# ---------------------------------------------------------------------
# 1) IPF / raking with hard margins
# ---------------------------------------------------------------------
def rake_to_margins(X: np.ndarray,
                    row_targets: np.ndarray,
                    col_targets: np.ndarray,
                    *,
                    tol: float = 1e-10,
                    max_iter: int = 1000,
                    denom_eps: float = 1e-12,
                    seed_eps: float = 1e-15) -> None:
    """
    In-place IPF so rows -> row_targets and cols -> col_targets.
    Adds tiny 'support seeding' so columns with positive targets aren't stuck at zero.
    """
    assert X.ndim == 2
    n, k = X.shape
    assert row_targets.shape == (n,)
    assert col_targets.shape == (k,)

    # ---- seed support: if both margins want positive mass but X==0, sprinkle tiny >0
    if seed_eps is not None and seed_eps > 0:
        want = (row_targets > 0)[:, None] & (col_targets > 0)[None, :]
        need = want & (X <= 0)
        if need.any():
            # proportional to outer product of margins (very tiny)
            R = row_targets / max(float(row_targets.sum()), 1.0)
            C = col_targets / max(float(col_targets.sum()), 1.0)
            X[need] = seed_eps * (R[:, None] * C[None, :])[need]

    # ---- IPF
    eps = denom_eps
    for _ in range(max_iter):
        # columns
        col_sums = X.sum(axis=0)
        cscale = np.divide(col_targets, np.maximum(col_sums, eps), where=np.isfinite(col_sums), out=np.ones_like(col_sums))
        X *= cscale  # broadcast over rows

        # rows
        row_sums = X.sum(axis=1, keepdims=True)
        rscale = np.divide(row_targets[:, None], np.maximum(row_sums, eps), where=np.isfinite(row_sums), out=np.ones_like(row_sums))
        X *= rscale  # broadcast over cols

        if (np.allclose(X.sum(axis=0), col_targets, rtol=1e-12, atol=1e-12) and
            np.allclose(X.sum(axis=1), row_targets, rtol=1e-12, atol=1e-12)):
            break



# ---------------------------------------------------------------------
# 2) Child totals adjuster (scalar per parent group)
# ---------------------------------------------------------------------

def make_child_totals_adj(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    parent_adj_col: str,   # parent-level HARD total to hit (e.g. "Insgesamt_*_adj" or raw "Insgesamt_*")
    child_total_col: str,  # child-level "Insgesamt_*" to be scaled (writes *_adj)
    out_col: Optional[str] = None
) -> pd.Series:
    """
    For each parent id, multiply child totals by a scalar so
    sum(children)_adj == parent_adj. Returns the new child series (also added as column).
    If a group's child sum is 0 but parent_adj > 0, split equally across its children.
    """
    out_col = out_col or f"{child_total_col}_adj"

    pids_parent = parent_df[parent_id_col].astype(str).str.strip()
    pids_child  = child_df[child_parent_id_col].astype(str).str.strip()

    parent_target = (parent_df
        .assign(_pid=pids_parent)
        .set_index("_pid")[parent_adj_col]
        .astype(float)
        .reindex(pids_child.unique())
        .fillna(0.0))

    child_tot = pd.to_numeric(child_df[child_total_col], errors="coerce").fillna(0.0)
    group_sum = pd.Series(child_tot.values, index=pids_child).groupby(level=0).sum().astype(float)

    scale = pd.Series(0.0, index=group_sum.index)
    mask_nz = group_sum > 0
    scale.loc[mask_nz] = parent_target.loc[mask_nz] / group_sum.loc[mask_nz]

    s = pids_child.map(scale).fillna(0.0)
    adj = child_tot * s

    # degenerate: group_sum==0 & parent_target>0  -> equal split (row-based, robust)
    group_sum_by_pid = group_sum  # (index = pid)
    parent_by_pid    = parent_target  # (index = pid)

    rows_deg = pids_child.map(group_sum_by_pid).le(0) & pids_child.map(parent_by_pid).gt(0)
    if rows_deg.any():
        counts_by_pid = pids_child.value_counts().reindex(parent_target.index).fillna(0).astype(int)
        per_cap_by_pid = (parent_target / counts_by_pid.replace(0, np.nan)).fillna(0.0)
        adj.loc[rows_deg] = pids_child.map(per_cap_by_pid).loc[rows_deg].to_numpy()


    child_df[out_col] = adj.values
    return child_df[out_col]


# ---------------------------------------------------------------------
# 3) Topic spec & downscaler
# ---------------------------------------------------------------------

@dataclass
class TopicSpec:
    """
    One categorical vector at a given level pair (parent→child).
    Provide:
      - parent category columns (order matters),
      - child  category columns (same order, same length),
      - child HARD row-total column (usually "Insgesamt_*", ideally *_adj).
    Optional:
      - alpha (damping),
      - blend (TrustBlend) for row weights w_i.
    """
    name: str
    parent_cat_cols: List[str]
    child_cat_cols:  List[str]
    child_row_total_col: str
    alpha: float = 0.85
    blend: TrustBlend = field(default_factory=TrustBlend)


def _assert_topic_columns(parent_df, child_df, parent_id_col, child_parent_id_col, spec: TopicSpec) -> None:
    missing_parent = [c for c in spec.parent_cat_cols if c not in parent_df.columns]
    missing_child  = [c for c in spec.child_cat_cols  if c not in child_df.columns]
    if missing_parent:
        raise KeyError(f"[{spec.name}] Missing parent cols: {missing_parent[:5]}{'...' if len(missing_parent)>5 else ''}")
    if missing_child:
        raise KeyError(f"[{spec.name}] Missing child cols: {missing_child[:5]}{'...' if len(missing_child)>5 else ''}")
    if spec.child_row_total_col not in child_df.columns:
        raise KeyError(f"[{spec.name}] Missing child total col: {spec.child_row_total_col}")
    if len(spec.parent_cat_cols) != len(spec.child_cat_cols):
        raise ValueError(f"[{spec.name}] Category length mismatch parent({len(spec.parent_cat_cols)}) "
                         f"vs child({len(spec.child_cat_cols)})")
    if parent_id_col not in parent_df.columns:
        raise KeyError(f"[{spec.name}] Missing parent_id_col: {parent_id_col}")
    if child_parent_id_col not in child_df.columns:
        raise KeyError(f"[{spec.name}] Missing child_parent_id_col: {child_parent_id_col}")


def _apply_topic_trust_blend(
    cdf: pd.DataFrame,
    X: np.ndarray,
    child_totals: np.ndarray,
    parent_shares: np.ndarray,
    child_cat_cols: List[str],
    alpha: float,
    blend: TrustBlend,
    eps: float = 1e-9
) -> None:
    """
    For each category k:
      target_i = w_i * local_ik + (1 - w_i) * (parent_share_k * child_total_i)
      Then scale X[:, k] by (target/s) ** alpha, with protections & seeding.
    """
    n, K = X.shape
    assert K == len(child_cat_cols)
    w = blend.weights(child_totals, K)

    for k, col in enumerate(child_cat_cols):
        local = pd.to_numeric(cdf[col], errors="coerce").fillna(0.0).to_numpy(float)
        nat_exp = parent_shares[k] * child_totals
        target  = w * local + (1.0 - w) * nat_exp

        s = X[:, k]
        need_seed = (s <= eps) & (target > 0)
        if need_seed.any():
            X[need_seed, k] = target[need_seed]
            s = X[:, k]

        good = (s > eps) & np.isfinite(s)
        f = np.ones_like(s)
        f[good] = (np.maximum(target[good], 0.0) / s[good]) ** alpha
        np.clip(f, 1e-6, 1e6, out=f)
        X[:, k] *= f


def downscale_topic(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    spec: TopicSpec,
    inner_passes: int = 10,
    outer_iters: int = 2,
    rake_tol: float = 1e-10,
    rake_max_iter: int = 50,
    validate_row_tol: float = 2e-4,
    verbose: bool = False
) -> pd.DataFrame:
    """Downscale one TopicSpec from parent -> child."""
    _assert_topic_columns(parent_df, child_df, parent_id_col, child_parent_id_col, spec)

    parent_df = parent_df.copy()
    child_df  = child_df.copy()
    parent_df[parent_id_col]      = parent_df[parent_id_col].astype(str).str.strip()
    child_df[child_parent_id_col] = child_df[child_parent_id_col].astype(str).str.strip()

    K = len(spec.child_cat_cols)
    out = pd.DataFrame(index=child_df.index, columns=spec.child_cat_cols, dtype=float)

    parent_map: Dict[object, np.ndarray] = {}
    for pid, psub in parent_df[[parent_id_col] + spec.parent_cat_cols].groupby(parent_id_col):
        parent_map[pid] = psub[spec.parent_cat_cols].astype(float).to_numpy().sum(axis=0)

    parent_sum_0_counter = 0
    diffcouter = 0
    for pid, cdf in tqdm(child_df.groupby(child_parent_id_col, sort=False, group_keys=False),
                         disable=not verbose):

        idx = cdf.index
        p = parent_map.get(pid)

        if p is None or cdf.empty:
            out.loc[idx, :] = 0.0
            continue

        t = pd.to_numeric(cdf[spec.child_row_total_col], errors="coerce").fillna(0.0).to_numpy(float)
        if t.size == 0:
            continue

        Psum = float(np.sum(p))
        if Psum == 0:
            parent_sum_0_counter += 1
            out.loc[idx, :] = 0.0
            continue
        if Psum < 1e-9:
            raise ValueError(f"[{spec.name}] Parent sum < 1e-9: {Psum}")

        # Mask tiny values to true 0
        mask_tiny = np.abs(p) < 1e-9
        if mask_tiny.any():
            p[mask_tiny] = 0.0
            if p.sum() > 0:
                p /= p.sum() / Psum  # rescale so total stays consistent


        Tsum = float(np.sum(t))
        if not np.isclose(Tsum, Psum, rtol=0, atol=1e-12):
            rel = abs(Psum - Tsum) / max(Psum, 1.0)
            if rel <= 1e-8:
                # tiny numeric drift -> silent rescale
                t *= Psum / max(Tsum, 1e-30)
            elif rel <= 1e-4:
                print(f"[warn] adjust row mass by {rel:.2e} for {pid}")
                t *= Psum / max(Tsum, 1e-30)
            else:
                raise AssertionError(
                    f"[fatal] infeasible margins (Δ={Psum-Tsum:.6g}, rel={rel:.2e}) for {pid}"
                )

        parent_shares = p / Psum
        zero_cols = parent_shares <= 0
        X = np.outer(t, parent_shares)

        for _ in range(outer_iters):
            for _inner in range(inner_passes):
                _apply_topic_trust_blend(
                    cdf, X, t, parent_shares, spec.child_cat_cols,
                    alpha=spec.alpha, blend=spec.blend
                )
                if zero_cols.any():
                    X[:, zero_cols] = 0.0  # ← keep pruned cols at 0

                # row re-projection
                rs = X.sum(axis=1, keepdims=True)
                np.divide(t[:, None], np.maximum(rs, 1e-12), out=rs)
                X *= rs

            # final rake (with support seeding, but only for positive cols)
            rake_to_margins(
                X, row_targets=t, col_targets=p,
                tol=rake_tol, max_iter=rake_max_iter,
                denom_eps=1e-12, seed_eps=1e-15
            )
            if zero_cols.any():
                X[:, zero_cols] = 0.0


        out.loc[idx, :] = X

        calc_rows = X.sum(axis=1)
        rel = np.abs(calc_rows - t) / np.maximum(t, 1.0)
        if rel.max(initial=0.0) > validate_row_tol:
            raise AssertionError(
                f"[{spec.name}] Row totals dev > {validate_row_tol:.2e} for parent {pid}. "
                f"max rel.err={rel.max():.3e}"
            )
        if not np.allclose(X.sum(axis=0), p, rtol=0, atol=1e-2):
            print("INFO: abs difference > 0.01")
            diffcouter += 1
            if not np.allclose(X.sum(axis=0), p, rtol=0, atol=1): # May get close for large or many child cells
                diff = X.sum(axis=0) - p
                rel = np.divide(diff, np.maximum(np.abs(p), 1e-12))
                print("WARNING: abs difference > 1")
                print(f"Parent ID: {pid}")
                print("Parent totals (p):")
                print(p)
                print("Child sums (X.sum(axis=0)):")
                print(X.sum(axis=0))
                print("Difference (child - parent):")
                print(diff)
                print("Relative difference:")
                print(rel)
                if not np.allclose(X.sum(axis=0), p, rtol=0.02, atol=0):
                    print("WARNING: rel difference > 0.02")

    print(f"Diffcouter: {diffcouter}")
    print(f"Parent sum 0: {parent_sum_0_counter} out of {len(parent_map)}")
    return out


# ---------------------------------------------------------------------
# 4) Helpers for level-specific column names (simple suffix swap)
# ---------------------------------------------------------------------

def levelize(cols_10km: List[str], level: str) -> List[str]:
    """
    Replace the trailing suffix *_10km-Gitter with *_{level}-Gitter
    level ∈ {"10km","1km","100m"}
    """
    from_str = "_10km-Gitter"
    to_str   = f"_{level}-Gitter"
    return [c.replace(from_str, to_str) for c in cols_10km]

# ---------------------------------------------------------------------
# 4.5) Normalize parent category vectors to their parent totals
# ---------------------------------------------------------------------

def _make_topic_prior_shares(df, cols):
    cols = [c for c in cols if c in df.columns]   # guard, though you already do this upstream
    if not cols:
        return np.full(1, 1.0)  # or return a uniform over K when you know K; see note below

    vals = df.loc[:, cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    w = vals.sum(axis=0).to_numpy(dtype=float)
    s = float(w.sum())
    if s > 0:
        return w / s
    return np.full(len(cols), 1.0 / len(cols), dtype=float)


def normalize_parent_categories_for_specs(
    parent_df: pd.DataFrame,
    *,
    specs: List[TopicSpec],
    child_level: str,            # '1km'
    cap_factor: float = 10.0,    # optional: flag overly large corrections
    verbose: bool = True,
) -> None:
    """
    For each TopicSpec:
      - compute a global prior over its parent category columns,
      - per parent row, scale cats so sum(cats) == parent total,
      - if sum(cats)==0 < and total>0, inject prior * total,
      - if total==0, set cats to 0.
    Operates IN-PLACE on parent_df.
    """
    for spec in specs:
        cat_cols = [c for c in spec.parent_cat_cols if c in parent_df.columns]
        if not cat_cols:
            if verbose:
                print(f"[norm] skip '{spec.name}': no parent category cols present")
            continue

        if child_level == "1km":
            tot_col = spec.child_row_total_col.replace("_1km-Gitter", "_10km-Gitter")
        # elif child_level == "100m":
        #     tot_col = spec.child_row_total_col.replace("_100m-Gitter", "_1km-Gitter") # NOT to be used at that level. Just for 10km init.
        else:
            raise ValueError(f"Unknown child_level: {child_level}")

        if tot_col not in parent_df.columns:
            if verbose:
                print(f"[norm] skip '{spec.name}': parent total '{tot_col}' not found")
            continue

        # Global prior shares for this topic
        prior = _make_topic_prior_shares(parent_df, cat_cols)  # (K,)

        # Numeric frames
        C = parent_df[cat_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)  # (N,K)
        T = pd.to_numeric(parent_df[tot_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)         # (N,)

        S = C.sum(axis=1)  # (N,)
        N, K = C.shape

        # Cases
        # 1) total == 0 -> set row to 0
        zero_tot = T <= 0
        if np.any(zero_tot):
            C[zero_tot, :] = 0.0

        # 2) sum(cats) == 0 & total > 0 -> inject prior * total
        need_inject = (S <= 0) & (T > 0)
        if np.any(need_inject):
            C[need_inject, :] = T[need_inject, None] * prior[None, :]

        # 3) baseline scaling for all rows with S > 0
        S = C.sum(axis=1)  # recompute after injection
        good = S > 0
        f = np.ones_like(S)
        f[good] = T[good] / S[good]

        # Optional: flag extreme factors
        if verbose:
            big = np.abs(f[good]) > cap_factor
            if np.any(big):
                n_big = int(np.sum(big))
                print(f"[norm] '{spec.name}' warning: {n_big} rows have |scale| > {cap_factor} (max={float(np.max(np.abs(f[good]))):.2f})")

        C *= f[:, None]

        # Write back
        parent_df.loc[:, cat_cols] = C
        if verbose:
            # cheap audit: check new sums vs totals
            new_s = C.sum(axis=1)
            err = np.abs(new_s - T) / np.maximum(T, 1.0)
            print(f"[norm] '{spec.name}' rows={len(err):,} | max rel.err={float(err.max()):.2e} | mean={float(err.mean()):.2e}")


# ---------------------------------------------------------------------
# 5) Configuration for PRIORITY topics
# ---------------------------------------------------------------------
# For 10-1 we did t_pc = 10
BLEND_STRONG = TrustBlend(w_min=0.50, t_pc=5.0)
BLEND_STD    = TrustBlend(w_min=0.40, t_pc=5.0)
BLEND_WEAK   = TrustBlend(w_min=0.30, t_pc=30.0)

def build_topic_specs_for_level(level: str) -> List[TopicSpec]:
    """
    Returns TopicSpec list for a given child level, assuming parent is the next coarser level:
      - for level="1km": parent is 10km
      - for level="100m": parent is 1km
    """
    # --- Familienstand (population by marital status) ---
    fam_tot_10 = "Insgesamt_Bevoelkerung_Familienstand_10km-Gitter"
    fam_cats_10 = [
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ]

    # --- Energieträger (population; NOT buildings-by-energy) ---
    et_tot_10 = "Insgesamt_Energietraeger_Energietraeger_10km-Gitter"
    et_cats_10 = [
        "Gas_Energietraeger_10km-Gitter",
        "Heizoel_Energietraeger_10km-Gitter",
        "Holz_Holzpellets_Energietraeger_10km-Gitter",
        "Biomasse_Biogas_Energietraeger_10km-Gitter",
        "Solar_Geothermie_Waermepumpen_Energietraeger_10km-Gitter",
        "Strom_Energietraeger_10km-Gitter",
        "Kohle_Energietraeger_10km-Gitter",
        "Fernwaerme_Energietraeger_10km-Gitter",
        "kein_Energietraeger_Energietraeger_10km-Gitter",
    ]

    # --- Heizungsart (Gebäude nach überwiegender Heizungsart) ---
    hz_tot_10 = "Insgesamt_Heizungsart_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter"
    hz_cats_10 = [
        "Fernheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Etagenheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Blockheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Zentralheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Einzel_Mehrraumoefen_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "keine_Heizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
    ]

    # --- Haushaltsgröße (size of private household) ---
    hh_tot_10 = "Insgesamt_Haushalte_Groesse_des_privaten_Haushalts_10km-Gitter"
    hh_cats_10 = [
        "1_Person_Groesse_des_privaten_Haushalts_10km-Gitter",
        "2_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "3_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "4_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "5_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "6_Personen_und_mehr_Groesse_des_privaten_Haushalts_10km-Gitter",
    ]

    # --- Lebensform (private HH life form) ---
    lf_tot_10 = "Insgesamt_Haushalte_Typ_priv_HH_Lebensform_10km-Gitter"
    lf_cats_10 = [
        "EinpersHH_SingleHH_Typ_priv_HH_Lebensform_10km-Gitter",
        "Ehepaare_Typ_priv_HH_Lebensform_10km-Gitter",
        "EingetrLebensp_Typ_priv_HH_Lebensform_10km-Gitter",
        "NichtehelLebensg_Typ_priv_HH_Lebensform_10km-Gitter",
        "AlleinerzMuetter_Typ_priv_HH_Lebensform_10km-Gitter",
        "AlleinerzVaeter_Typ_priv_HH_Lebensform_10km-Gitter",
        "MehrpersHHohneKernfam_Typ_priv_HH_Lebensform_10km-Gitter",
    ]

    # --- Räume (Wohnungen nach Zahl der Räume) ---
    rm_tot_10 = "Insgesamt_Wohnungen_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter"
    rm_cats_10 = [
        "1Raum_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "2Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "3Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "4Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "5Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "6Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "7undmehrRaeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
    ]

    # --- Wohnungsfläche (10 m² intervals) ---
    fl_tot_10 = "Insgesamt_Wohnungen_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter"
    fl_cats_10 = [
        "unter30_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "30bis39_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "40bis49_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "50bis59_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "60bis69_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "70bis79_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "80bis89_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "90bis99_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "100bis109_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "110bis119_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "120bis129_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "130bis139_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "140bis149_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "150bis159_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "160bis169_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "170bis179_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "180undmehr_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
    ]

    # --- Geburtsland (groups) ---
    gl_tot_10 = "Insgesamt_Bevoelkerung_Geburtsland_Gruppen_10km-Gitter"
    gl_cats_10 = [
        "Deutschland_Geburtsland_Gruppen_10km-Gitter",
        "Ausland_Sonstige_Geburtsland_Gruppen_10km-Gitter",
        "EU27_Land_Geburtsland_Gruppen_10km-Gitter",
        "Sonstiges_Europa_Geburtsland_Gruppen_10km-Gitter",
        "Sonstige_Welt_Geburtsland_Gruppen_10km-Gitter",
        "Sonstige_Geburtsland_Gruppen_10km-Gitter",
    ]

    def topic(name, tot_10, cats_10, alpha, blend):
        if level == "1km":
            parent_cols = cats_10
            child_cols  = levelize(cats_10, "1km")
            child_total = tot_10.replace("_10km-Gitter", "_1km-Gitter")
        elif level == "100m":
            parent_cols = levelize(cats_10, "1km")
            child_cols  = levelize(cats_10, "100m")
            child_total = tot_10.replace("_10km-Gitter", "_100m-Gitter")
        else:
            raise ValueError(f"Unknown level: {level}")

        return TopicSpec(
            name=name,
            parent_cat_cols=parent_cols,
            child_cat_cols=child_cols,
            child_row_total_col=child_total,
            alpha=alpha,
            blend=blend,
        )

    specs = [
        topic("Familienstand",    fam_tot_10, fam_cats_10, alpha=0.90, blend=BLEND_STD),
        topic("Energietraeger",   et_tot_10,  et_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Heizungsart",      hz_tot_10,  hz_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Haushaltsgroesse", hh_tot_10,  hh_cats_10,  alpha=0.90, blend=BLEND_STD),
        topic("Lebensform",       lf_tot_10,  lf_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Raeume",           rm_tot_10,  rm_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Wohnflaeche",      fl_tot_10,  fl_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Geburtsland",      gl_tot_10,  gl_cats_10,  alpha=0.85, blend=BLEND_STD),
    ]
    return specs



# ---------------------------------------------------------------------
# 5.5) Shim: create *_adj child totals for ALL topics and flip specs
# ---------------------------------------------------------------------


def _require_parent_adj_for_child_total(parent_df: pd.DataFrame, child_total_col: str) -> str:
    if child_total_col.endswith("_100m-Gitter"):
        base = child_total_col.replace("_100m-Gitter", "_1km-Gitter")
        adj = f"{base}_adj"
        if adj not in parent_df.columns:
            raise KeyError(f"Missing required parent total: {adj} for {child_total_col}")
        return adj
    if child_total_col.endswith("_1km-Gitter"):
        return child_total_col.replace("_1km-Gitter", "_10km-Gitter")
    return child_total_col


def apply_adj_for_all_topics(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    specs: List[TopicSpec],
    verbose: bool = True,
) -> List[TopicSpec]:
    """For each TopicSpec, create a *_adj child row-total and switch the spec to use it."""
    specs_out: List[TopicSpec] = []
    for spec in specs:
        total_col = spec.child_row_total_col
        adj_col = f"{total_col}_adj"
        if adj_col not in child_df.columns:
            parent_total = _require_parent_adj_for_child_total(parent_df, total_col)
            make_child_totals_adj(
                parent_df=parent_df,
                child_df=child_df,
                parent_id_col=parent_id_col,
                child_parent_id_col=child_parent_id_col,
                parent_adj_col=parent_total,
                child_total_col=total_col,
                out_col=adj_col,
            )
            if verbose:
                print(f"[adj] Created {adj_col} for topic '{spec.name}'")
        specs_out.append(TopicSpec(
            name=spec.name,
            parent_cat_cols=spec.parent_cat_cols,
            child_cat_cols=spec.child_cat_cols,
            child_row_total_col=adj_col,
            alpha=spec.alpha,
            blend=spec.blend,
        ))
    return specs_out


def impute_orphan_rows_100m(
    df: pd.DataFrame,
    *,
    specs: List[TopicSpec],
    orphan_flag_col: str = "is_orphan",
    dtype_out = np.float32,
    eps: float = 1e-12,
    verbose: bool = True,
) -> None:
    """
    For 100m rows marked as orphans:
      - If the row has category signal, scale to its *_adj* total.
      - Else allocate the total using a prior from non-orphan 100m category sums.

    Operates IN-PLACE on `df`.
    """
    if orphan_flag_col not in df.columns:
        if verbose:
            print(f"[orphans] '{orphan_flag_col}' not found; nothing to do.")
        return

    mask_orphan = df[orphan_flag_col].to_numpy(bool)
    mask_non    = ~mask_orphan
    n_orph = int(mask_orphan.sum())
    if n_orph == 0:
        if verbose:
            print("[orphans] no orphan rows.")
        return

    for spec in specs:
        # Ensure columns exist
        cats = [c for c in spec.child_cat_cols if c in df.columns]
        totc = spec.child_row_total_col
        if not cats or totc not in df.columns:
            if verbose:
                print(f"[orphans:{spec.name}] skip (missing cols).")
            continue

        # Build prior from non-orphans (sum over computed 100m results)
        prior_counts = (
            df.loc[mask_non, cats]
              .apply(pd.to_numeric, errors="coerce")
              .fillna(0.0)
              .sum(axis=0)
              .to_numpy(dtype=float)
        )
        prior = prior_counts + eps
        psum  = float(prior.sum())
        if psum <= 0:
            # fallback uniform if absolutely no mass
            prior = np.full(len(cats), 1.0 / max(len(cats), 1), dtype=float)
        else:
            prior = prior / psum  # normalized prior

        # Orphan rows: read totals + current cat signals
        T = pd.to_numeric(df.loc[mask_orphan, totc], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        Y = df.loc[mask_orphan, cats].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)

        # Classify
        S = Y.sum(axis=1)
        caseA = (S > eps) & (T > 0)   # has signal -> scale to total
        caseB = (S <= eps) & (T > 0)  # no signal -> use prior
        caseZ = (T <= 0)              # zero total -> all zeros

        Xo = np.zeros_like(Y, dtype=float)

        # Case A: preserve shape, scale to T
        if np.any(caseA):
            scale = T[caseA] / np.maximum(S[caseA], eps)
            Xo[caseA, :] = Y[caseA, :] * scale[:, None]

        # Case B: use prior
        if np.any(caseB):
            Xo[caseB, :] = T[caseB, None] * prior[None, :]

        # Case Z already zeros

        # Hygiene: clip tiny negatives, cast, and write back
        np.maximum(Xo, 0.0, out=Xo)
        df.loc[mask_orphan, cats] = Xo.astype(dtype_out)

        if verbose:
            # quick audits
            rows = int(np.sum(mask_orphan))
            used_A = int(np.sum(caseA))
            used_B = int(np.sum(caseB))
            used_Z = int(np.sum(caseZ))
            # row-sum check (aggregate)
            rs = Xo.sum(axis=1)
            rel = np.abs(rs - T) / np.maximum(T, 1.0)
            print(f"[orphans:{spec.name}] rows={rows} | A={used_A} B={used_B} Z={used_Z} "
                  f"| max row rel.err={float(rel.max()):.3e} | mean={float(rel.mean()):.3e}")

# ---------------------------------------------------------------------
# 6) Run
# ---------------------------------------------------------------------

if __name__ == "__main__":
    # Paths
    PATH_10  = r"T:\petre\UCFL\Synthetic Population\Zensus\merged\df10_with_single_years.pickle"
    PATH_1   = r"T:\petre\UCFL\Synthetic Population\Zensus\merged\df1_with_single_years.pickle"
    PATH_100 = r"T:\petre\UCFL\Synthetic Population\Zensus\with_gender\cells_100m_with_gender_backfilled.parquet"

    OUT_1    = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_1km_with_binneds.parquet"
    OUT_100  = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_100m_with_gender_backf_binneds.parquet"

    # Load pickles (reset index so IDs become normal cols)
    df10 = pd.read_pickle(PATH_10).reset_index(drop=False)
    df1  = pd.read_pickle(PATH_1).reset_index(drop=False)
    #
    # Clean NaNs/±inf (already done before, but eh)
    for df in (df10, df1):
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.fillna(0, inplace=True)

    # Drop werterlaeuternde_Zeichen as they just clutter up the data
    df10 = df10.drop(columns=[c for c in df10.columns if "werterlaeuternde_Zeichen" in c])
    df1 = df1.drop(columns=[c for c in df1.columns if "werterlaeuternde_Zeichen" in c])


    # Normalize IDs
    df10["GITTER_ID_10km"] = df10["GITTER_ID_10km"].astype(str).str.strip()
    df1["GITTER_ID_10km"]  = df1["GITTER_ID_10km"].astype(str).str.strip()
    df1["GITTER_ID_1km"]   = df1["GITTER_ID_1km"].astype(str).str.strip()

    # -------------------------
    # 10km → 1km
    # -------------------------
    topics_1km = build_topic_specs_for_level("1km")

    # Normalize 10km category vectors to their *Insgesamt* totals, needed bcs they must match, but very often don't bcs of perturbation
    # (bit weird function naming. Parent just means that later 10km will be parent of 1km)
    normalize_parent_categories_for_specs(
        parent_df=df10,
        specs=topics_1km,
        child_level="1km",
        verbose=True
    )

    topics_1km = apply_adj_for_all_topics(
        parent_df=df10, child_df=df1,
        parent_id_col="GITTER_ID_10km", child_parent_id_col="GITTER_ID_10km",
        specs=topics_1km, verbose=True
    )


    for spec in tqdm(topics_1km, desc="Topics 1km"):
        res = downscale_topic(
            parent_df=df10, child_df=df1,
            parent_id_col="GITTER_ID_10km",
            child_parent_id_col="GITTER_ID_10km",
            spec=spec,
            inner_passes=10, outer_iters=2,
            rake_tol=1e-11, rake_max_iter=1000,
            validate_row_tol=2e-4, verbose=False
        )
        for c in spec.child_cat_cols:
            df1[c] = res[c].values

    df1.to_parquet(OUT_1, index=False)
    df1 = pd.read_parquet(OUT_1)

    # -------------------------
    # 1km → 100m (memory-safe, full-output)
    # -------------------------
    topics_100m = build_topic_specs_for_level("100m")

    # Build minimal column set needed to compute topics at 100m
    needed_cols_100 = {"GITTER_ID_1km"}
    for spec in topics_100m:
        needed_cols_100.add(spec.child_row_total_col)
        needed_cols_100.update(spec.child_cat_cols)
    needed_cols_100 = sorted(needed_cols_100)

    # Load ONLY needed columns (keeps original row order) and reset index
    df100_min = pd.read_parquet(PATH_100, columns=needed_cols_100).reset_index(drop=False)

    # Clean + downcast for RAM
    df100_min.replace([np.inf, -np.inf], np.nan, inplace=True)
    df100_min.fillna(0, inplace=True)
    for c in df100_min.columns:
        if pd.api.types.is_float_dtype(df100_min[c]):
            df100_min[c] = df100_min[c].astype(np.float32)
    df100_min["GITTER_ID_1km"] = df100_min["GITTER_ID_1km"].astype(str).str.strip()

    # link coverage flags
    p_1km = set(df1["GITTER_ID_1km"].unique())
    df100_min["is_orphan"] = ~df100_min["GITTER_ID_1km"].isin(p_1km)
    df100_ok = df100_min.loc[~df100_min["is_orphan"]].copy()
    df1_ok   = df1.loc[df1["GITTER_ID_1km"].isin(df100_ok["GITTER_ID_1km"])].copy()

    # DEBUG TEMP -----------------------------------------------------------
    # PID = "CRS3035RES1000mN2775000E4437000"; df100_ok = df100_min.loc[(~df100_min["is_orphan"]) & (df100_min["GITTER_ID_1km"]==PID)].copy(); df1_ok = df1.loc[df1["GITTER_ID_1km"]==PID].copy()
    # -----------------------------------------------------------------

    # Create *_adj and flip specs to use them
    topics_100m = apply_adj_for_all_topics(
        parent_df=df1_ok, child_df=df100_ok,
        parent_id_col="GITTER_ID_1km", child_parent_id_col="GITTER_ID_1km",
        specs=topics_100m, verbose=True
    )
    adj_total_cols = [spec.child_row_total_col for spec in topics_100m]  # these are now *_adj
    for col in adj_total_cols:
        if col in df100_ok.columns:
            df100_min.loc[df100_ok.index, col] = df100_ok[col].astype(np.float32).values

    # Compute downscaled categories and place back into df100_min at matching positions
    for spec in tqdm(topics_100m, desc="Topics 100m"):
        res = downscale_topic(
            parent_df=df1_ok, child_df=df100_ok,
            parent_id_col="GITTER_ID_1km",
            child_parent_id_col="GITTER_ID_1km",
            spec=spec,
            inner_passes=10, outer_iters=2,
            rake_tol=1e-11, rake_max_iter=1000,
            validate_row_tol=2e-4, verbose=False
        )
        for c in spec.child_cat_cols:
            df100_min.loc[df100_ok.index, c] = res[c].values.astype(np.float32)

    # --- Handle 100m orphans: fit to total using 100m prior (or local shape if present)
    # Nope, we do that separately.
    # impute_orphan_rows_100m(
    #     df=df100_min,
    #     specs=topics_100m,
    #     orphan_flag_col="is_orphan",
    #     dtype_out=np.float32,
    #     eps=1e-12,
    #     verbose=True,
    # )


    # --- Write FULL output parquet: stream original + append new columns by position ---
    # Drop werterlaeuternde_Zeichen as in 10 and 1 dfs
    dataset = ds.dataset(PATH_100, format="parquet")

    drop_pat = "werterlaeuternde_Zeichen"
    orig_names = dataset.schema.names
    drop_cols_100 = [c for c in orig_names if drop_pat in c]
    keep_cols_100 = [c for c in orig_names if c not in drop_cols_100]

    # write both *_adj totals and per-category columns
    new_cols = []
    for spec in topics_100m:
        new_cols.append(spec.child_row_total_col)    # *_adj total
        new_cols.extend(spec.child_cat_cols)         # per-category results

    for c in new_cols:
        if c not in df100_min.columns:
            df100_min[c] = np.nan

    # Build writer schema: original (without the dropped cols) + extra fields
    orig_schema = dataset.schema
    base_fields = [f for f in orig_schema if f.name in keep_cols_100]
    extra_fields = [pa.field(c, pa.float32()) for c in new_cols if c not in keep_cols_100]
    full_schema = pa.schema(list(base_fields) + extra_fields)

    writer = None
    pos = 0
    batch_size = 1_000_000

    # IMPORTANT: scan only keep_cols_100 so dropped columns never enter the pipeline
    scanner = dataset.scanner(columns=keep_cols_100, batch_size=batch_size)
    reader = scanner.to_reader()

    for rb in reader:
        tbl = pa.Table.from_batches([rb])
        n = tbl.num_rows

        combined = tbl

        # append/replace computed arrays from df100_min slice
        for c in new_cols:
            # choose type: existing column -> keep file's type; new column -> float32
            if c in combined.schema.names:
                expected_type = combined.schema.field(c).type          # usually double
            else:
                expected_type = pa.float32()                           # our new computed cols

            arr = pa.array(df100_min[c].iloc[pos:pos+n].to_numpy(), type=expected_type)

            if c in combined.schema.names:
                idx = combined.schema.get_field_index(c)
                combined = combined.set_column(idx, c, arr)
            else:
                combined = combined.append_column(c, arr)

        # enforce exact column order to match the writer schema
        combined = combined.select([f.name for f in full_schema])

        if writer is None:
            writer = pq.ParquetWriter(OUT_100, full_schema)

        writer.write_table(combined)
        pos += n

    if writer:
        writer.close()



    # -------------------------
    # quick echoes (use the small frames)
    # -------------------------
    def _echo(df, cols, total_col, tag):
        if total_col not in df.columns:
            print(f"[{tag}] skipped: total column '{total_col}' not found.")
            return
        calc = df[cols].sum(axis=1).to_numpy(float)
        given = pd.to_numeric(df[total_col], errors="coerce").fillna(0.0).to_numpy(float)
        rel = np.abs(calc - given) / np.maximum(given, 1.0)
        print(f"[{tag}] rows={len(df):,} | max row rel.err={rel.max():.3e} | mean={rel.mean():.3e}")

    # Echo (Familienstand)
    fam1_cols = levelize([
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ], "1km")
    _echo(df1, fam1_cols,
          "Insgesamt_Bevoelkerung_Familienstand_1km-Gitter_adj",
          "1km Familienstand")

    fam100_cols = levelize([
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ], "100m")
    # echo on the small frame (df100_min) which now holds the computed columns
    _echo(df100_min, fam100_cols,
          "Insgesamt_Bevoelkerung_Familienstand_100m-Gitter_adj",
          "100m Familienstand")


[norm] 'Familienstand' rows=3,824 | max rel.err=4.76e-16 | mean=7.80e-17
[norm] 'Energietraeger' rows=3,824 | max rel.err=5.67e-16 | mean=8.08e-17
[norm] 'Heizungsart' rows=3,824 | max rel.err=3.86e-16 | mean=5.16e-17
[norm] 'Haushaltsgroesse' rows=3,824 | max rel.err=3.97e-16 | mean=6.62e-17
[norm] 'Lebensform' rows=3,824 | max rel.err=4.10e-16 | mean=6.77e-17
[norm] 'Raeume' rows=3,824 | max rel.err=3.93e-16 | mean=3.77e-17
[norm] 'Wohnflaeche' rows=3,824 | max rel.err=5.87e-16 | mean=8.18e-17
[norm] 'Geburtsland' rows=3,824 | max rel.err=4.19e-16 | mean=7.26e-17
[adj] Created Insgesamt_Bevoelkerung_Familienstand_1km-Gitter_adj for topic 'Familienstand'
[adj] Created Insgesamt_Energietraeger_Energietraeger_1km-Gitter_adj for topic 'Energietraeger'
[adj] Created Insgesamt_Heizungsart_Gebaeude_nach_ueberwiegender_Heizungsart_1km-Gitter_adj for topic 'Heizungsart'
[adj] Created Insgesamt_Haushalte_Groesse_des_privaten_Haushalts_1km-Gitter_adj for topic 'Haushaltsgroesse'
[adj] Created I

Topics 1km:  12%|█▎        | 1/8 [00:45<05:21, 46.00s/it]

Diffcouter: 0
Parent sum 0: 2 out of 3824


Topics 1km:  25%|██▌       | 2/8 [01:43<05:15, 52.57s/it]

Diffcouter: 0
Parent sum 0: 4 out of 3824


...

INFO: abs difference > 0.01
INFO: abs difference > 0.01
INFO: abs difference > 0.01
Diffcouter: 3
Parent sum 0: 15469 out of 211444


Topics 100m:  88%|████████▊ | 7/8 [6:20:06<56:34, 3394.20s/it]  

Diffcouter: 0
Parent sum 0: 1629 out of 211444


Topics 100m: 100%|██████████| 8/8 [6:51:46<00:00, 3088.29s/it]


[1km Familienstand] rows=212,758 | max row rel.err=1.103e-15 | mean=1.579e-16
 Familienstand] rows=3,148,482 | max row rel.err=6.000e+00 | mean=2.481e-05


In [2]:
# Fix orphans after the fact (as it didn't properly work in the full run, and that takes hours). Orphans are a very tiny fraction of all 100m cells, and an even tinier part of the data (typically very sparse in orphans), but we still care about them :)

import numpy as np
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from dataclasses import dataclass, field
from typing import Dict, List, Optional


def _topic_cols_for_100m(specs):
    topics = []
    for spec in specs:
        cats = list(spec.child_cat_cols)
        totc = spec.child_row_total_col
        # prefer *_adj if present in file; we resolve existence per-batch below
        topics.append((spec.name, cats, totc))
    return topics

def post_fix_orphans_100m_stream(
    path_in: str,
    path_out: str,
    *,
    specs,
    orphan_flag_col: str = "is_orphan",
    batch_size: int = 1_000_000,
    eps: float = 1e-12,
    verbose: bool = True,
):
    dataset = ds.dataset(path_in, format="parquet")
    schema  = dataset.schema

    # ---- prep topics (lists of cat cols + total col name)
    topics = _topic_cols_for_100m(specs)

    # ---- PASS 1: accumulate non-orphan priors per topic (streaming sums)
    if verbose: print("[post-fix] pass 1: building non-orphan priors...")
    prior_sums = {name: None for name, _, _ in topics}
    non_orphan_count = 0
    # read only needed columns this pass
    cols_needed_pass1 = {orphan_flag_col}
    for _, cats, totc in topics:
        cols_needed_pass1.update(cats)
        cols_needed_pass1.add(totc)
        # also add base total if totc endswith _adj; we’ll pick whichever exists
        if totc.endswith("_adj"):
            cols_needed_pass1.add(totc[:-4])
    cols_needed_pass1 = [c for c in cols_needed_pass1 if c in schema.names]

    for rb in dataset.to_batches(columns=cols_needed_pass1, batch_size=batch_size):
        tbl = pa.Table.from_batches([rb])
        if orphan_flag_col not in tbl.schema.names:
            if verbose: print(f"[post-fix] '{orphan_flag_col}' not found; nothing to do.")
            return
        is_orphan = tbl[orphan_flag_col].to_numpy(zero_copy_only=False)
        non_orphan = ~is_orphan
        nn = int(non_orphan.sum())
        non_orphan_count += nn
        if nn == 0:
            continue

        for name, cats, totc in topics:
            cats_present = [c for c in cats if c in tbl.schema.names]
            if not cats_present:
                continue
            # sum over non-orphans
            chunk = np.zeros(len(cats_present), dtype=np.float64)
            for j, c in enumerate(cats_present):
                arr = tbl[c].to_numpy(zero_copy_only=False)
                # robust numeric: nan->0
                v = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
                chunk[j] = v[non_orphan].sum(dtype=np.float64)
            if prior_sums[name] is None:
                prior_sums[name] = (cats_present, chunk)
            else:
                prev_cols, prev_sum = prior_sums[name]
                # handle case differing cat presence across batches
                if prev_cols != cats_present:
                    # unify ordering by intersection
                    inter = [c for c in cats_present if c in prev_cols]
                    if not inter:
                        continue
                    # reduce both to intersection
                    idx_prev = [prev_cols.index(c) for c in inter]
                    idx_curr = [cats_present.index(c) for c in inter]
                    prev_sum = prev_sum[idx_prev]
                    chunk    = chunk[idx_curr]
                    prior_sums[name] = (inter, prev_sum + chunk)
                else:
                    prior_sums[name] = (prev_cols, prev_sum + chunk)

    priors = {}
    for name, cats, _ in topics:
        info = prior_sums.get(name)
        if info is None:
            continue
        cats_present, sums = info
        s = float(sums.sum())
        if s <= eps:
            pri = np.full(len(cats_present), 1.0 / max(len(cats_present), 1), dtype=np.float64)
        else:
            pri = (sums / s).astype(np.float64)
        priors[name] = (cats_present, pri)
        if verbose:
            print(f"[post-fix] prior for '{name}': cats={len(cats_present)} | mass={s:.3f}")

    if verbose:
        print(f"[post-fix] non-orphan rows seen: {non_orphan_count:,}")

    # ---- PASS 2: rewrite file with repaired orphans for topic cat columns
    if verbose: print("[post-fix] pass 2: rewriting with fixes...")
    writer = None
    try:
        # we’ll write row groups equal to incoming batches
        for rb in dataset.to_batches(batch_size=batch_size):
            tbl = pa.Table.from_batches([rb])
            n = tbl.num_rows
            if n == 0:
                continue

            # pull orphan mask once
            if orphan_flag_col not in tbl.schema.names:
                # no orphan col in this batch (unlikely) -> write through
                out_tbl = tbl
            else:
                is_orphan = tbl[orphan_flag_col].to_numpy(zero_copy_only=False)
                if np.any(is_orphan):
                    # for each topic, repair orphan rows in its cat columns
                    arrays_new = {}
                    for name, cats_all, totc in topics:
                        if name not in priors:  # no usable prior
                            continue
                        cats_present, prior = priors[name]
                        # restrict to columns that exist in this batch
                        cats_batch = [c for c in cats_present if c in tbl.schema.names]
                        if not cats_batch:
                            continue

                        # choose total col in this batch
                        totc_batch = totc if totc in tbl.schema.names else (totc[:-4] if totc.endswith("_adj") and totc[:-4] in tbl.schema.names else None)
                        if totc_batch is None:
                            # fall back: totals = sum(cats)
                            T = None
                        else:
                            T = tbl[totc_batch].to_numpy(zero_copy_only=False)
                            T = np.nan_to_num(T, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64, copy=False)

                        # build Y (current values) for these cats
                        Y = np.column_stack([
                            np.nan_to_num(tbl[c].to_numpy(zero_copy_only=False), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64, copy=False)
                            for c in cats_batch
                        ]) if cats_batch else None

                        if Y is None:
                            continue

                        # compute fixes only for orphan rows
                        # --- build a prior vector aligned to cats_batch (replaces any ad-hoc fallback)
                        cats_prior, prior_full = priors[name]
                        prior_map = {c: prior_full[i] for i, c in enumerate(cats_prior)}
                        p = np.array([prior_map.get(c, 0.0) for c in cats_batch], dtype=np.float64)
                        p_sum = p.sum()
                        if p_sum <= eps:
                            p[:] = 1.0 / len(cats_batch)
                        else:
                            p /= p_sum

                        # --- compute fixes only for orphan rows (same logic, but we'll write into writable copies)
                        idx = is_orphan
                        if not np.any(idx):
                            continue

                        Y_orph = Y[idx, :]
                        S = Y_orph.sum(axis=1)
                        To = (Y.sum(axis=1) if T is None else T)[idx]

                        caseA = (S > eps) & (To > 0)
                        caseB = (S <= eps) & (To > 0)
                        caseZ = (To <= 0)

                        Xo = np.zeros_like(Y_orph, dtype=np.float64)
                        if np.any(caseA):
                            Xo[caseA, :] = Y_orph[caseA, :] * (To[caseA] / np.maximum(S[caseA], eps))[:, None]
                        if np.any(caseB):
                            Xo[caseB, :] = To[caseB, None] * p[None, :]

                        # ---- WRITE BACK (make writable copies to avoid "read-only" numpy buffer)
                        for j, c in enumerate(cats_batch):
                            col = tbl.column(c)
                            ftype = schema.field(c).type  # preserve original type
                            # make a WRITABLE numpy copy before in-place assignment
                            arr = np.array(col.combine_chunks().to_numpy(zero_copy_only=False),
                                           dtype=np.float64, copy=True)
                            arr[idx] = Xo[:, j]
                            # cast back to original (float32 or float64)
                            if pa.types.is_float32(ftype):
                                arrays_new[c] = pa.array(arr.astype(np.float32, copy=False), type=pa.float32())
                            else:
                                arrays_new[c] = pa.array(arr.astype(np.float64, copy=False), type=pa.float64())

                        if verbose:
                            max_rel = 0.0
                            if To.size:
                                rs = Xo.sum(axis=1)
                                rel = np.abs(rs - To) / np.maximum(To, 1.0)
                                max_rel = float(rel.max(initial=0.0))
                            print(f"[post-fix:{name}] batch orphans={int(idx.sum())} | "
                                  f"A={int(caseA.sum())} B={int(caseB.sum())} Z={int(caseZ.sum())} | "
                                  f"max row rel.err={max_rel:.3e}")


                    # materialize replacements into a new table
                    cols = []
                    for f in schema:
                        name = f.name
                        if name in arrays_new:
                            cols.append(arrays_new[name])
                        else:
                            cols.append(tbl[name])
                    out_tbl = pa.Table.from_arrays(cols, schema=schema)
                else:
                    out_tbl = tbl  # no orphans in this batch

            # open writer if needed
            if writer is None:
                writer = pq.ParquetWriter(path_out, schema)
            writer.write_table(out_tbl)

    finally:
        if writer is not None:
            if verbose:
                print(f"[post-fix] wrote: {path_out}")
            writer.close()


def build_topic_specs_for_level(level: str):
    """
    Returns TopicSpec list for a given child level, assuming parent is the next coarser level:
      - for level="1km": parent is 10km
      - for level="100m": parent is 1km
    """
    # --- Familienstand (population by marital status) ---
    fam_tot_10 = "Insgesamt_Bevoelkerung_Familienstand_10km-Gitter"
    fam_cats_10 = [
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ]

    # --- Energieträger (population; NOT buildings-by-energy) ---
    et_tot_10 = "Insgesamt_Energietraeger_Energietraeger_10km-Gitter"
    et_cats_10 = [
        "Gas_Energietraeger_10km-Gitter",
        "Heizoel_Energietraeger_10km-Gitter",
        "Holz_Holzpellets_Energietraeger_10km-Gitter",
        "Biomasse_Biogas_Energietraeger_10km-Gitter",
        "Solar_Geothermie_Waermepumpen_Energietraeger_10km-Gitter",
        "Strom_Energietraeger_10km-Gitter",
        "Kohle_Energietraeger_10km-Gitter",
        "Fernwaerme_Energietraeger_10km-Gitter",
        "kein_Energietraeger_Energietraeger_10km-Gitter",
    ]

    # --- Heizungsart (Gebäude nach überwiegender Heizungsart) ---
    hz_tot_10 = "Insgesamt_Heizungsart_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter"
    hz_cats_10 = [
        "Fernheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Etagenheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Blockheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Zentralheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Einzel_Mehrraumoefen_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "keine_Heizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
    ]

    # --- Haushaltsgröße (size of private household) ---
    hh_tot_10 = "Insgesamt_Haushalte_Groesse_des_privaten_Haushalts_10km-Gitter"
    hh_cats_10 = [
        "1_Person_Groesse_des_privaten_Haushalts_10km-Gitter",
        "2_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "3_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "4_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "5_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "6_Personen_und_mehr_Groesse_des_privaten_Haushalts_10km-Gitter",
    ]

    # --- Lebensform (private HH life form) ---
    lf_tot_10 = "Insgesamt_Haushalte_Typ_priv_HH_Lebensform_10km-Gitter"
    lf_cats_10 = [
        "EinpersHH_SingleHH_Typ_priv_HH_Lebensform_10km-Gitter",
        "Ehepaare_Typ_priv_HH_Lebensform_10km-Gitter",
        "EingetrLebensp_Typ_priv_HH_Lebensform_10km-Gitter",
        "NichtehelLebensg_Typ_priv_HH_Lebensform_10km-Gitter",
        "AlleinerzMuetter_Typ_priv_HH_Lebensform_10km-Gitter",
        "AlleinerzVaeter_Typ_priv_HH_Lebensform_10km-Gitter",
        "MehrpersHHohneKernfam_Typ_priv_HH_Lebensform_10km-Gitter",
    ]

    # --- Räume (Wohnungen nach Zahl der Räume) ---
    rm_tot_10 = "Insgesamt_Wohnungen_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter"
    rm_cats_10 = [
        "1Raum_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "2Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "3Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "4Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "5Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "6Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "7undmehrRaeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
    ]

    # --- Wohnungsfläche (10 m² intervals) ---
    fl_tot_10 = "Insgesamt_Wohnungen_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter"
    fl_cats_10 = [
        "unter30_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "30bis39_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "40bis49_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "50bis59_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "60bis69_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "70bis79_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "80bis89_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "90bis99_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "100bis109_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "110bis119_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "120bis129_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "130bis139_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "140bis149_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "150bis159_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "160bis169_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "170bis179_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "180undmehr_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
    ]

    # --- Geburtsland (groups) ---
    gl_tot_10 = "Insgesamt_Bevoelkerung_Geburtsland_Gruppen_10km-Gitter"
    gl_cats_10 = [
        "Deutschland_Geburtsland_Gruppen_10km-Gitter",
        "Ausland_Sonstige_Geburtsland_Gruppen_10km-Gitter",
        "EU27_Land_Geburtsland_Gruppen_10km-Gitter",
        "Sonstiges_Europa_Geburtsland_Gruppen_10km-Gitter",
        "Sonstige_Welt_Geburtsland_Gruppen_10km-Gitter",
        "Sonstige_Geburtsland_Gruppen_10km-Gitter",
    ]

    def topic(name, tot_10, cats_10, alpha, blend):
        if level == "1km":
            parent_cols = cats_10
            child_cols  = levelize(cats_10, "1km")
            child_total = tot_10.replace("_10km-Gitter", "_1km-Gitter")
        elif level == "100m":
            parent_cols = levelize(cats_10, "1km")
            child_cols  = levelize(cats_10, "100m")
            child_total = tot_10.replace("_10km-Gitter", "_100m-Gitter")
        else:
            raise ValueError(f"Unknown level: {level}")

        return TopicSpec(
            name=name,
            parent_cat_cols=parent_cols,
            child_cat_cols=child_cols,
            child_row_total_col=child_total,
            alpha=alpha,
            blend=blend,
        )

    specs = [
        topic("Familienstand",    fam_tot_10, fam_cats_10, alpha=0.90, blend=BLEND_STD),
        topic("Energietraeger",   et_tot_10,  et_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Heizungsart",      hz_tot_10,  hz_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Haushaltsgroesse", hh_tot_10,  hh_cats_10,  alpha=0.90, blend=BLEND_STD),
        topic("Lebensform",       lf_tot_10,  lf_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Raeume",           rm_tot_10,  rm_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Wohnflaeche",      fl_tot_10,  fl_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Geburtsland",      gl_tot_10,  gl_cats_10,  alpha=0.85, blend=BLEND_STD),
    ]
    return specs


def levelize(cols_10km, level: str):
    """
    Replace the trailing suffix *_10km-Gitter with *_{level}-Gitter
    level ∈ {"10km","1km","100m"}
    """
    from_str = "_10km-Gitter"
    to_str   = f"_{level}-Gitter"
    return [c.replace(from_str, to_str) for c in cols_10km]

@dataclass
class TrustBlend:
    """
    Compute local trust weight w_i in [w_min, 1] from child row totals T_i.
    - w_min : baseline trust even for tiny totals
    - t_pc  : "people per category" target; when T_i ≈ K * t_pc, w_i ~ 1
              (K = number of categories in the topic)
    """
    w_min: float = 0.40
    t_pc: float  = 5.0

    def weights(self, totals: np.ndarray, n_categories: int) -> np.ndarray:
        cutoff = max(self.t_pc * max(n_categories, 1), 1e-9)
        base = np.minimum(1.0, totals / cutoff)
        return self.w_min + (1.0 - self.w_min) * base


@dataclass
class TrustBlend:
    w_min: float = 0.40
    t_pc: float  = 10.0

@dataclass
class TopicSpec:
    name: str
    parent_cat_cols: List[str]
    child_cat_cols:  List[str]
    child_row_total_col: str
    alpha: float = 0.85
    blend: TrustBlend = field(default_factory=TrustBlend)

# For 10-1 we did t_pc = 10
BLEND_STRONG = TrustBlend(w_min=0.50, t_pc=5.0)
BLEND_STD    = TrustBlend(w_min=0.40, t_pc=5.0)
BLEND_WEAK   = TrustBlend(w_min=0.30, t_pc=30.0)

PATH_100 = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_100m_with_gender_backf_binneds.parquet"
OUT_100_FIXED = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_100m_with_gender_backf_binneds_happyorphans.parquet"
topics_100m = build_topic_specs_for_level("100m") # From cell above
post_fix_orphans_100m_stream(PATH_100, OUT_100_FIXED, specs=topics_100m, orphan_flag_col="is_orphan", batch_size=1_000_000, verbose=True)


[post-fix] pass 1: building non-orphan priors...
[post-fix] prior for 'Familienstand': cats=8 | mass=82708633.702
[post-fix] prior for 'Energietraeger': cats=9 | mass=43102969.179
[post-fix] prior for 'Heizungsart': cats=6 | mass=19953844.853
[post-fix] prior for 'Haushaltsgroesse': cats=6 | mass=40233755.820
[post-fix] prior for 'Lebensform': cats=7 | mass=40233755.820
[post-fix] prior for 'Raeume': cats=7 | mass=41803914.300
[post-fix] prior for 'Wohnflaeche': cats=17 | mass=41803914.302
[post-fix] prior for 'Geburtsland': cats=6 | mass=82708633.701
[post-fix] non-orphan rows seen: 3,148,437
[post-fix] pass 2: rewriting with fixes...
[post-fix:Familienstand] batch orphans=13 | A=8 B=4 Z=1 | max row rel.err=0.000e+00
[post-fix:Energietraeger] batch orphans=13 | A=2 B=0 Z=11 | max row rel.err=0.000e+00
[post-fix:Heizungsart] batch orphans=13 | A=0 B=0 Z=13 | max row rel.err=0.000e+00
[post-fix:Haushaltsgroesse] batch orphans=13 | A=1 B=0 Z=12 | max row rel.err=0.000e+00
[post-fix:Leben

In [3]:
# Sanity check thoroughly.

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
from typing import List, Tuple, Dict

# --- configure ---
PATH_100_FIXED = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_100m_with_gender_backf_binneds_happyorphans.parquet"
PATH_1KM       = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_1km_with_binneds.parquet"

# Choose which totals to use in checks:
#   ""     -> use raw "Insgesamt_*_Gitter"
#   "_adj" -> use adjusted "Insgesamt_*_Gitter_adj"
TOTAL_SUFFIX_100M = "_adj"   # switch to "" to audit raw 100m
TOTAL_SUFFIX_1KM  = "_adj"       # switch to "_adj" if you wrote *_adj at 1km for topics
POP_TOTAL_SUFFIX_100M = "_adj"  # for age POP_TOTAL_100m vs POP_TOTAL_100m_adj

# cell-level tolerances (for per-value comparisons)
ABS_TOL = 1e-2
REL_TOL = 2e-2

# row-level tolerances (for row-wise topic vectors)
ROW_LINF_THR   = 0.5         # flag if any category differs by > 0.5
ROW_L1REL_THR  = 1.0e-2      # or if L1 / max(total,1) > 1%

BATCH = 1_000_000

# ---------------- helpers ----------------

def _print_worst_offenders(df_agg: pd.DataFrame,
                           df1_sub: pd.DataFrame,
                           tot100: str, tot1: str,
                           cats_pairs: List[Tuple[str, str]],
                           topn: int = 10) -> None:
    """
    Print the worst 'topn' 1km parents by |child_agg - parent| total.
    Two lines per offender:
      1) grid id + total diff (abs/rel) and both totals,
      2) top category diffs (by absolute diff), a compact list.
    """
    if tot100 not in df_agg.columns or tot1 not in df1_sub.columns:
        print("    (skip worst-offenders: missing totals)")
        return

    parent_tot = _safe_num(df1_sub[tot1].to_numpy())
    child_tot  = _safe_num(df_agg[tot100].to_numpy())
    d_tot = child_tot - parent_tot

    if d_tot.size == 0:
        print("    (skip worst-offenders: no rows)")
        return

    # indices of largest absolute differences
    order = np.argsort(np.abs(d_tot))[-topn:][::-1]
    ids = df1_sub.index.to_numpy()

    for i in order:
        gid = ids[i]
        p_tot = float(parent_tot[i])
        c_tot = float(child_tot[i])
        diff  = float(d_tot[i])
        rel   = diff / (p_tot if p_tot != 0 else 1.0)

        # line 1: totals
        print(f"    {gid}: Δtot={diff:+.3f} (child={c_tot:.3f}, parent={p_tot:.3f}, rel={rel:.3e})")

        # line 2: top category contributors
        rows = []
        for c100, c1k in cats_pairs:
            if c100 in df_agg.columns and c1k in df1_sub.columns:
                dc = float(_safe_num(df_agg.at[gid, c100])) - float(_safe_num(df1_sub.at[gid, c1k]))
                rows.append((c1k, dc))
        if rows:
            rows.sort(key=lambda x: abs(x[1]), reverse=True)
            # keep it compact; top 5 usually enough
            topcats = ", ".join([f"{name}:{d:+.2f}" for name, d in rows[:5]])
            print(f"      top cat Δ: {topcats}")
        else:
            print("      top cat Δ: (no comparable category columns)")

def _age_columns():
    return [f"AGE_{i}" for i in range(0, 101)]

def _age_sum(a, b):
    return [f"AGE_{i}" for i in range(a, b+1)]

def _age_checks_10yr():
    return [
        ("Unter10_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(0,9)),
        ("a10bis19_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(10,19)),
        ("a20bis29_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(20,29)),
        ("a30bis39_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(30,39)),
        ("a40bis49_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(40,49)),
        ("a50bis59_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(50,59)),
        ("a60bis69_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(60,69)),
        ("a70bis79_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(70,79)),
        ("a80undaelter_Alter_in_10er-Jahresgruppen_100m-Gitter", _age_sum(80,100)),
    ]

def _age_checks_5cls():
    return [
        ("Unter18_Alter_in_5_Altersklassen_100m-Gitter", _age_sum(0,17)),
        ("a18bis29_Alter_in_5_Altersklassen_100m-Gitter", _age_sum(18,29)),
        ("a30bis49_Alter_in_5_Altersklassen_100m-Gitter", _age_sum(30,49)),
        ("a50bis64_Alter_in_5_Altersklassen_100m-Gitter", _age_sum(50,64)),
        ("a65undaelter_Alter_in_5_Altersklassen_100m-Gitter", _age_sum(65,100)),
    ]

def _topic_sets_base():
    fam = ("Insgesamt_Bevoelkerung_Familienstand", [
        "Ledig_Familienstand","Verheiratet_Familienstand","Verwitwet_Familienstand",
        "Geschieden_Familienstand","EingetrLebenspartnerschaft_Familienstand",
        "EingetrLebenspartVerstorben_Familienstand","EingetrLebenspartAufgehoben_Familienstand",
        "OhneAngabe_Familienstand",
    ])
    et = ("Insgesamt_Energietraeger_Energietraeger", [
        "Gas_Energietraeger","Heizoel_Energietraeger","Holz_Holzpellets_Energietraeger",
        "Biomasse_Biogas_Energietraeger","Solar_Geothermie_Waermepumpen_Energietraeger",
        "Strom_Energietraeger","Kohle_Energietraeger","Fernwaerme_Energietraeger",
        "kein_Energietraeger_Energietraeger",
    ])
    hz = ("Insgesamt_Heizungsart_Gebaeude_nach_ueberwiegender_Heizungsart", [
        "Fernheizung_Gebaeude_nach_ueberwiegender_Heizungsart",
        "Etagenheizung_Gebaeude_nach_ueberwiegender_Heizungsart",
        "Blockheizung_Gebaeude_nach_ueberwiegender_Heizungsart",
        "Zentralheizung_Gebaeude_nach_ueberwiegender_Heizungsart",
        "Einzel_Mehrraumoefen_Gebaeude_nach_ueberwiegender_Heizungsart",
        "keine_Heizung_Gebaeude_nach_ueberwiegender_Heizungsart",
    ])
    hh = ("Insgesamt_Haushalte_Groesse_des_privaten_Haushalts", [
        "1_Person_Groesse_des_privaten_Haushalts","2_Personen_Groesse_des_privaten_Haushalts",
        "3_Personen_Groesse_des_privaten_Haushalts","4_Personen_Groesse_des_privaten_Haushalts",
        "5_Personen_Groesse_des_privaten_Haushalts","6_Personen_und_mehr_Groesse_des_privaten_Haushalts",
    ])
    lf = ("Insgesamt_Haushalte_Typ_priv_HH_Lebensform", [
        "EinpersHH_SingleHH_Typ_priv_HH_Lebensform","Ehepaare_Typ_priv_HH_Lebensform",
        "EingetrLebensp_Typ_priv_HH_Lebensform","NichtehelLebensg_Typ_priv_HH_Lebensform",
        "AlleinerzMuetter_Typ_priv_HH_Lebensform","AlleinerzVaeter_Typ_priv_HH_Lebensform",
        "MehrpersHHohneKernfam_Typ_priv_HH_Lebensform",
    ])
    rm = ("Insgesamt_Wohnungen_Wohnungen_nach_Zahl_der_Raeume", [
        "1Raum_Wohnungen_nach_Zahl_der_Raeume","2Raeume_Wohnungen_nach_Zahl_der_Raeume",
        "3Raeume_Wohnungen_nach_Zahl_der_Raeume","4Raeume_Wohnungen_nach_Zahl_der_Raeume",
        "5Raeume_Wohnungen_nach_Zahl_der_Raeume","6Raeume_Wohnungen_nach_Zahl_der_Raeume",
        "7undmehrRaeume_Wohnungen_nach_Zahl_der_Raeume",
    ])
    fl = ("Insgesamt_Wohnungen_Flaeche_der_Wohnung_10m2_Intervalle", [
        "unter30_Flaeche_der_Wohnung_10m2_Intervalle","30bis39_Flaeche_der_Wohnung_10m2_Intervalle",
        "40bis49_Flaeche_der_Wohnung_10m2_Intervalle","50bis59_Flaeche_der_Wohnung_10m2_Intervalle",
        "60bis69_Flaeche_der_Wohnung_10m2_Intervalle","70bis79_Flaeche_der_Wohnung_10m2_Intervalle",
        "80bis89_Flaeche_der_Wohnung_10m2_Intervalle","90bis99_Flaeche_der_Wohnung_10m2_Intervalle",
        "100bis109_Flaeche_der_Wohnung_10m2_Intervalle","110bis119_Flaeche_der_Wohnung_10m2_Intervalle",
        "120bis129_Flaeche_der_Wohnung_10m2_Intervalle","130bis139_Flaeche_der_Wohnung_10m2_Intervalle",
        "140bis149_Flaeche_der_Wohnung_10m2_Intervalle","150bis159_Flaeche_der_Wohnung_10m2_Intervalle",
        "160bis169_Flaeche_der_Wohnung_10m2_Intervalle","170bis179_Flaeche_der_Wohnung_10m2_Intervalle",
        "180undmehr_Flaeche_der_Wohnung_10m2_Intervalle",
    ])
    gl = ("Insgesamt_Bevoelkerung_Geburtsland_Gruppen", [
        "Deutschland_Geburtsland_Gruppen","Ausland_Sonstige_Geburtsland_Gruppen",
        "EU27_Land_Geburtsland_Gruppen","Sonstiges_Europa_Geburtsland_Gruppen",
        "Sonstige_Welt_Geburtsland_Gruppen","Sonstige_Geburtsland_Gruppen",
    ])
    return [fam, et, hz, hh, lf, rm, fl, gl]

def _topic_cols_100m(topic, total_suffix="") -> Tuple[str, List[str], List[str]]:
    tot_base, cats_base = topic
    tot_raw = f"{tot_base}_100m-Gitter"
    tot = tot_raw + total_suffix
    cats = [f"{c}_100m-Gitter" for c in cats_base]
    return tot, cats, [tot] + cats

def _topic_cols_1km(topic, total_suffix="") -> Tuple[str, List[str]]:
    tot_base, cats_base = topic
    tot_raw = f"{tot_base}_1km-Gitter"
    tot = tot_raw + total_suffix
    cats = [f"{c}_1km-Gitter" for c in cats_base]
    return tot, cats

def _safe_num(a):
    return np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)

def _raw_adj_info(ds_any, col_raw, suffix="_adj", batch=BATCH) -> Tuple[bool, Dict[str, float]]:
    """
    If both col_raw and col_raw+suffix exist in schema, compute a small summary:
      - rows compared
      - bad rows (abs/rel tol with the same ABS_TOL/REL_TOL)
      - global mass child(adj)-raw and relative
      - max |diff|
    Returns (has_both, stats_dict)
    """
    schema = ds_any.schema
    col_adj = col_raw + suffix
    if col_raw not in schema.names or col_adj not in schema.names:
        return False, {}
    rows = 0
    bad = 0
    max_abs = 0.0
    sum_raw = 0.0
    sum_adj = 0.0
    for rb in ds_any.to_batches(columns=[col_raw, col_adj], batch_size=batch):
        tbl = pa.Table.from_batches([rb])
        if tbl.num_rows == 0: continue
        rows += tbl.num_rows
        r = _safe_num(tbl[col_raw].to_numpy(zero_copy_only=False))
        a = _safe_num(tbl[col_adj].to_numpy(zero_copy_only=False))
        d = a - r
        ok = (np.abs(d) <= ABS_TOL) | (np.abs(d) <= REL_TOL * np.maximum(np.abs(r), 1.0))
        bad += int((~ok).sum())
        if d.size:
            max_abs = max(max_abs, float(np.max(np.abs(d))))
        sum_raw += float(r.sum())
        sum_adj += float(a.sum())
    diff = sum_adj - sum_raw
    rel  = diff / (sum_raw if sum_raw != 0 else 1.0)
    return True, {
        "rows": rows, "bad": bad, "max_abs": max_abs,
        "sum_raw": sum_raw, "sum_adj": sum_adj, "diff": diff, "rel": rel
    }
# ===================== RICHER SANITY CHECK =====================

# --- switches: which totals to use ---
TOT_SUFFIX_100M = "_adj"   # "" to use raw 100m totals; "_adj" to use *_adj totals
TOT_SUFFIX_1KM  = "_adj"   # "" or "_adj" for 1km totals
POP_TOTAL_SUFFIX = "_adj"  # "" or "_adj" for POP_TOTAL_100m

# --- thresholds/bins for detailed reporting ---
AGE_GATE = 50.0  # split age-aggregate checks by target >= AGE_GATE vs < AGE_GATE

ABS_BINS_CELL = [0.25, 0.5, 1, 2, 5, 10]     # for |diff| at cell-level
REL_BINS_CELL = [0.01, 0.02, 0.05, 0.10, 0.20, 0.50]  # for |diff| / denom

L_INF_BINS = [0.5, 1, 2, 5, 10]   # for row-wise L∞
L1_REL_BINS = [0.01, 0.02, 0.05, 0.10, 0.20]  # for row-wise L1 / total


def _resolve_total(col_base: str, level_suffix: str, want_suffix: str, schema_names) -> str:
    """
    col_base like 'Insgesamt_Bevoelkerung_Familienstand', level_suffix like '_100m-Gitter'.
    If want_suffix is '_adj', prefer '<base><level>_adj' when present; otherwise fallback to raw '<base><level>'.
    If want_suffix is '', prefer raw; if missing, try '_adj'.
    """
    raw = f"{col_base}{level_suffix}"
    adj = f"{raw}_adj"
    if want_suffix == "_adj":
        if adj in schema_names: return adj
        if raw in schema_names: return raw
    else:
        if raw in schema_names: return raw
        if adj in schema_names: return adj
    return raw  # last resort (may be missing; caller should guard)


def _hist_counts(x: np.ndarray, bins: List[float]) -> List[int]:
    if x.size == 0: return [0] * (len(bins) + 1)
    b = np.asarray(bins, dtype=float)
    idx = np.digitize(x, b, right=True)  # 0..len(bins)
    return [int((idx == i).sum()) for i in range(len(b))] + [int((idx == len(b)).sum())]


def _fmt_hist(bins: List[float], counts: List[int], label: str) -> str:
    segs = []
    prev = None
    for i, edge in enumerate(bins):
        if i == 0:
            segs.append(f"[≤{edge}]:{counts[i]:,}")
        else:
            segs.append(f"({bins[i-1]},{edge}]:{counts[i]:,}")
        prev = edge
    segs.append(f">(last):{counts[-1]:,}")
    return f"{label} " + " | ".join(segs)


def sanity_check(path_100_fixed: str,
                 path_1km: str,
                 abs_tol=ABS_TOL,
                 rel_tol=REL_TOL,
                 row_linf_thr=ROW_LINF_THR,
                 row_l1rel_thr=ROW_L1REL_THR,
                 batch=BATCH):

    print(f"Using totals: 100m='{TOT_SUFFIX_100M}', 1km='{TOT_SUFFIX_1KM}'; Age POP total='{POP_TOTAL_SUFFIX}'.\n")

    ds100 = ds.dataset(path_100_fixed, format="parquet")
    schema100 = ds100.schema

    # ---------- 0) Ages: internal ----------
    ages = _age_columns()

    pop_col = _resolve_total("POP_TOTAL_100m", "", POP_TOTAL_SUFFIX, schema100.names)
    need_age = ["GITTER_ID_1km", pop_col] + ages + [
        "Insgesamt_Bevoelkerung_Alter_in_10er-Jahresgruppen_100m-Gitter",
        "Insgesamt_Bevoelkerung_Alter_in_5_Altersklassen_100m-Gitter",
    ] + [n for n,_ in _age_checks_10yr()] + [n for n,_ in _age_checks_5cls()]
    need_age = [c for c in need_age if c in schema100.names]

    total_rows = 0
    bad_pop_sum = 0
    max_pop_abs = 0.0
    max_pop_rel = 0.0

    # age aggregate accumulators (two groups: target >= AGE_GATE vs < AGE_GATE)
    agg_stats = {
        "big": dict(n=0, bad=0, sum_abs=0.0, max_abs=0.0, abs_hist=[0]*(len(ABS_BINS_CELL)+1),
                    sum_rel=0.0, max_rel=0.0, rel_hist=[0]*(len(REL_BINS_CELL)+1)),
        "small": dict(n=0, bad=0, sum_abs=0.0, max_abs=0.0, abs_hist=[0]*(len(ABS_BINS_CELL)+1),
                      sum_rel=0.0, max_rel=0.0, rel_hist=[0]*(len(REL_BINS_CELL)+1)),
    }
    # optional: track which named aggregate contributes most "bad" (top 5)
    bad_by_name_big: Dict[str,int] = {}
    bad_by_name_small: Dict[str,int] = {}

    def _accumulate_group(name: str, target: np.ndarray, diff: np.ndarray, group_key: str):
        g = agg_stats[group_key]
        # tolerances use denom = max(target, 1) like your other checks
        denom = np.maximum(target, 1.0)
        ad = np.abs(diff)
        rd = ad / denom

        # flags by tol
        ok = (ad <= abs_tol) | (ad <= rel_tol * denom)
        n = ad.size
        g["n"] += n
        g["bad"] += int((~ok).sum())
        g["sum_abs"] += float(ad.sum())
        g["max_abs"] = max(g["max_abs"], float(ad.max())) if n else g["max_abs"]
        g["sum_rel"] += float(rd.sum())
        g["max_rel"] = max(g["max_rel"], float(rd.max())) if n else g["max_rel"]

        # histograms
        ah = _hist_counts(ad, ABS_BINS_CELL)
        rh = _hist_counts(rd, REL_BINS_CELL)
        g["abs_hist"] = [int(a+b) for a,b in zip(g["abs_hist"], ah)]
        g["rel_hist"] = [int(a+b) for a,b in zip(g["rel_hist"], rh)]

        # per-aggregate-name bad tally
        if n:
            if group_key == "big":
                bad_by_name_big[name] = bad_by_name_big.get(name, 0) + int((~ok).sum())
            else:
                bad_by_name_small[name] = bad_by_name_small.get(name, 0) + int((~ok).sum())

    # streaming over 100m
    for rb in ds100.to_batches(columns=need_age, batch_size=batch):
        tbl = pa.Table.from_batches([rb])
        n = tbl.num_rows
        if n == 0:
            continue
        total_rows += n

        # POP total vs sum(age)
        if pop_col in tbl.schema.names and all(a in tbl.schema.names for a in ages):
            pop = _safe_num(tbl[pop_col].to_numpy(zero_copy_only=False))
            A = np.column_stack([_safe_num(tbl[a].to_numpy(zero_copy_only=False)) for a in ages]).sum(axis=1)
            diff = A - pop
            abs_ok = np.abs(diff) <= abs_tol
            rel_ok = np.abs(diff) <= rel_tol * np.maximum(pop, 1.0)
            ok = abs_ok | rel_ok
            bad_pop_sum += int((~ok).sum())
            if diff.size:
                max_pop_abs = max(max_pop_abs, float(np.max(np.abs(diff))))
                max_pop_rel = max(max_pop_rel, float(np.max(np.abs(diff) / np.maximum(pop, 1.0))))

        # aggregate (10y & 5cls) vs per-age
        def _acc(name, bucket):
            if name in tbl.schema.names and all(b in tbl.schema.names for b in bucket):
                target = _safe_num(tbl[name].to_numpy(zero_copy_only=False))
                S = np.column_stack([_safe_num(tbl[b].to_numpy(zero_copy_only=False)) for b in bucket]).sum(axis=1)
                d = S - target
                m_big = target >= AGE_GATE
                m_sml = (target > 0) & (target < AGE_GATE)

                if m_big.any():
                    _accumulate_group(name, target[m_big], d[m_big], "big")
                if m_sml.any():
                    _accumulate_group(name, target[m_sml], d[m_sml], "small")

        for name, bucket in _age_checks_10yr(): _acc(name, bucket)
        for name, bucket in _age_checks_5cls(): _acc(name, bucket)

    print("\n[100m internal age checks]")
    print(f"rows seen: {total_rows:,}")
    print(f"{pop_col} ≟ ∑AGE_0..100 -> bad rows: {bad_pop_sum:,} ({100*bad_pop_sum/max(total_rows,1):.4f}%) | max abs={max_pop_abs:.3e} | max rel={max_pop_rel:.3e}")

    def _dump_age_group(tag, g):
        n = max(g['n'], 1)
        mean_abs = g['sum_abs'] / n
        mean_rel = g['sum_rel'] / n
        print(f"Aggregates (10y & 5cls) vs per-age [{tag}]")
        print(f"  cells={g['n']:,} | mean |diff|={mean_abs:.4f} | max |diff|={g['max_abs']:.3e} "
              f"| mean rel.err={mean_rel:.4e} | max rel.err={g['max_rel']:.3e} | bad by tol={g['bad']:,}")
        print("  " + _fmt_hist(ABS_BINS_CELL, g['abs_hist'], "abs bins:"))
        print("  " + _fmt_hist(REL_BINS_CELL, g['rel_hist'], "rel bins:"))

    _dump_age_group(f"target≥{int(AGE_GATE)}", agg_stats["big"])
    _dump_age_group(f"target<{int(AGE_GATE)}", agg_stats["small"])

    # if both POP_TOTAL raw & adj exist, compare them
    pop_raw = "POP_TOTAL_100m"
    pop_adj = "POP_TOTAL_100m_adj"
    if pop_raw in schema100.names and pop_adj in schema100.names:
        # stream quick compare
        rows = 0; bad = 0; maxdiff = 0.0; sum_raw = 0.0; sum_adj = 0.0 # Bad here just means a sig. diff, not actually "bad"
        for rb in ds100.to_batches(columns=[pop_raw, pop_adj], batch_size=batch):
            tbl = pa.Table.from_batches([rb])
            if tbl.num_rows == 0: continue
            a = _safe_num(tbl[pop_raw].to_numpy(zero_copy_only=False))
            b = _safe_num(tbl[pop_adj].to_numpy(zero_copy_only=False))
            d = a - b
            rows += d.size
            bad += int((np.abs(d) > abs_tol).sum())
            if d.size: maxdiff = max(maxdiff, float(np.max(np.abs(d))))
            sum_raw += float(a.sum()); sum_adj += float(b.sum())
        diff = sum_raw - sum_adj
        rel = diff / (sum_adj if sum_adj != 0 else 1.0)
        print(f"{pop_raw} raw↔adj: rows={rows:,} | sig.diff.rows={bad:,} | max|diff|={maxdiff:.3e} | global raw={sum_raw:.3f} adj={sum_adj:.3f} diff={diff:.3e} ({rel:.3e} rel)")

    # ---------- 1) Topics: internal row totals at 100m ----------
    topics = _topic_sets_base()
    print("\n[100m topic-internal totals (∑cats ≟ total)]")
    for tot_base, cats_base in topics:
        tot100_raw = f"{tot_base}_100m-Gitter"
        tot100 = _resolve_total(tot_base, "_100m-Gitter", TOT_SUFFIX_100M, schema100.names)
        cats100 = [f"{c}_100m-Gitter" for c in cats_base]

        if tot100 not in schema100.names or any(c not in schema100.names for c in cats100):
            print(f"- {tot100}: skipped (missing total or categories in 100m)")
            continue

        nrows = bad_rows = 0
        max_abs = max_rel = 0.0
        # hist for row diffs
        abs_hist = [0]*(len(ABS_BINS_CELL)+1)
        rel_hist = [0]*(len(REL_BINS_CELL)+1)
        sum_abs = sum_rel = 0.0

        for rb in ds100.to_batches(columns=[tot100] + cats100, batch_size=batch):
            tbl = pa.Table.from_batches([rb])
            if tbl.num_rows == 0: continue
            nrows += tbl.num_rows
            T = _safe_num(tbl[tot100].to_numpy(zero_copy_only=False))
            S = np.column_stack([_safe_num(tbl[c].to_numpy(zero_copy_only=False)) for c in cats100]).sum(axis=1)
            d = S - T
            denom = np.maximum(T, 1.0)
            ok = (np.abs(d) <= abs_tol) | (np.abs(d) <= rel_tol * denom)
            bad_rows += int((~ok).sum())

            ad = np.abs(d); rd = ad / denom
            sum_abs += float(ad.sum()); sum_rel += float(rd.sum())
            if d.size:
                max_abs = max(max_abs, float(ad.max()))
                max_rel = max(max_rel, float(rd.max()))
            # hist
            ah = _hist_counts(ad, ABS_BINS_CELL)
            rh = _hist_counts(rd, REL_BINS_CELL)
            abs_hist = [a+b for a,b in zip(abs_hist, ah)]
            rel_hist = [a+b for a,b in zip(rel_hist, rh)]

        pct = 100.0 * bad_rows / max(nrows,1)
        mean_abs = (sum_abs / max(nrows,1))
        mean_rel = (sum_rel / max(nrows,1))
        print(f"- {tot100}: rows={nrows:,} | bad={bad_rows:,} ({pct:.4f}%) | max abs={max_abs:.3e} | max rel={max_rel:.3e} | mean |diff|={mean_abs:.4f} | mean rel.err={mean_rel:.4e}")
        print("    " + _fmt_hist(ABS_BINS_CELL, abs_hist, "abs bins:"))
        print("    " + _fmt_hist(REL_BINS_CELL, rel_hist, "rel bins:"))

        # raw vs adj totals info if both exist
        tot100_adj = f"{tot100_raw}_adj"
        if tot100_raw in schema100.names and tot100_adj in schema100.names:
            rows = bad = 0; maxdiff = 0.0; s_raw = s_adj = 0.0
            for rb in ds100.to_batches(columns=[tot100_raw, tot100_adj], batch_size=batch):
                tbl = pa.Table.from_batches([rb])
                if tbl.num_rows == 0: continue
                a = _safe_num(tbl[tot100_raw].to_numpy(zero_copy_only=False))
                b = _safe_num(tbl[tot100_adj].to_numpy(zero_copy_only=False))
                d = a - b
                rows += d.size
                bad += int((np.abs(d) > abs_tol).sum())
                if d.size: maxdiff = max(maxdiff, float(np.max(np.abs(d))))
                s_raw += float(a.sum()); s_adj += float(b.sum())
            dd = s_raw - s_adj; rr = dd / (s_adj if s_adj != 0 else 1.0)
            print(f"  info raw↔adj: rows={rows:,} | bad={bad:,} | max|diff|={maxdiff:.3e} | global raw={s_raw:.3f} adj={s_adj:.3f} diff={dd:.3e} ({rr:.3e} rel)")

    # ---------- 2) 1km topic-internal totals ----------
    ds1 = ds.dataset(path_1km, format="parquet")

    def _load_1km_subset(cols: List[str]) -> pd.DataFrame:
        need = list(set(["GITTER_ID_1km"] + [c for c in cols]))
        t = ds1.to_table(columns=[c for c in need if c in ds1.schema.names])
        if t.num_rows == 0:
            return pd.DataFrame(columns=["GITTER_ID_1km"]).set_index("GITTER_ID_1km")
        df = t.to_pandas(types_mapper=pd.ArrowDtype)
        return df.set_index("GITTER_ID_1km", drop=False)

    print("\n[1km topic-internal totals (∑cats ≟ total)]")
    for tot_base, cats_base in topics:
        tot1_raw = f"{tot_base}_1km-Gitter"
        tot1 = _resolve_total(tot_base, "_1km-Gitter", TOT_SUFFIX_1KM, ds1.schema.names)
        cats1 = [f"{c}_1km-Gitter" for c in cats_base]
        df1_sub = _load_1km_subset([tot1] + cats1)
        if tot1 not in df1_sub.columns or any(c not in df1_sub.columns for c in cats1):
            print(f"- {tot1}: skipped (missing total or categories in 1km)")
            continue

        T = _safe_num(df1_sub[tot1].to_numpy())
        S = _safe_num(df1_sub[cats1].to_numpy()).sum(axis=1)
        d = S - T
        denom = np.maximum(T, 1.0)
        ok = (np.abs(d) <= abs_tol) | (np.abs(d) <= rel_tol * denom)
        bad = int((~ok).sum())
        pct = 100.0 * bad / max(len(T),1)
        ad = np.abs(d); rd = ad / denom
        max_abs = float(ad.max()) if d.size else 0.0
        max_rel = float(rd.max()) if d.size else 0.0
        mean_abs = float(ad.mean()) if d.size else 0.0
        mean_rel = float(rd.mean()) if d.size else 0.0
        abs_hist = _hist_counts(ad, ABS_BINS_CELL)
        rel_hist = _hist_counts(rd, REL_BINS_CELL)
        print(f"- {tot1}: rows={len(T):,} | bad={bad:,} ({pct:.4f}%) | max abs={max_abs:.3e} | max rel={max_rel:.3e} | mean |diff|={mean_abs:.4f} | mean rel.err={mean_rel:.4e}")
        print("    " + _fmt_hist(ABS_BINS_CELL, abs_hist, "abs bins:"))
        print("    " + _fmt_hist(REL_BINS_CELL, rel_hist, "rel bins:"))

        # raw vs adj info if both exist
        tot1_adj = f"{tot1_raw}_adj"
        have_raw = tot1_raw in df1_sub.columns
        have_adj = tot1_adj in df1_sub.columns
        if have_raw and have_adj:
            a = _safe_num(df1_sub[tot1_raw].to_numpy())
            b = _safe_num(df1_sub[tot1_adj].to_numpy())
            d = a - b
            bad = int((np.abs(d) > abs_tol).sum())
            maxd = float(np.max(np.abs(d))) if d.size else 0.0
            s_raw = float(a.sum()); s_adj = float(b.sum())
            dd = s_raw - s_adj; rr = dd / (s_adj if s_adj != 0 else 1.0)
            print(f"  info raw↔adj: rows={len(d):,} | bad={bad:,} | max|diff|={maxd:.3e} | global raw={s_raw:.0f} adj={s_adj:.0f} diff={dd:.3e} ({rr:.3e} rel)")

    # ---------- 3) 100m→1km comparisons (row-wise + global mass) ----------
    print("\n[100m→1km topic comparisons (row-wise + global mass)]")
    for tot_base, cats_base in topics:
        tot100 = _resolve_total(tot_base, "_100m-Gitter", TOT_SUFFIX_100M, schema100.names)
        tot1   = _resolve_total(tot_base, "_1km-Gitter",  TOT_SUFFIX_1KM,  ds1.schema.names)
        cats100 = [f"{c}_100m-Gitter" for c in cats_base]
        cats1   = [f"{c}_1km-Gitter"  for c in cats_base]

        cols_exist = [c for c in (["GITTER_ID_1km"] + [tot100] + cats100) if c in schema100.names]
        if "GITTER_ID_1km" not in cols_exist:
            print(f"- {tot1}: skipped (GITTER_ID_1km missing in 100m)")
            continue

        # aggregate 100m -> 1km
        agg: Dict[str, pd.Series] = {}
        for rb in ds100.to_batches(columns=cols_exist, batch_size=batch):
            tbl = pa.Table.from_batches([rb])
            if tbl.num_rows == 0: continue
            key = tbl["GITTER_ID_1km"].to_numpy(zero_copy_only=False)
            for c in [col for col in [tot100] + cats100 if col in tbl.schema.names]:
                v = _safe_num(tbl[c].to_numpy(zero_copy_only=False))
                dfb = pd.DataFrame({"key": key, c: v})
                s = dfb.groupby("key", sort=False)[c].sum()
                agg[c] = s if c not in agg else agg[c].add(s, fill_value=0.0)

        if not agg:
            print(f"- {tot1}: skipped (no columns present at 100m)")
            continue

        df1_sub = _load_1km_subset([tot1] + cats1)
        if df1_sub.empty:
            print(f"- {tot1}: skipped (1km subset empty)")
            continue

        # align
        df_agg = pd.DataFrame(index=df1_sub.index)
        for c, s in agg.items():
            df_agg[c] = s.reindex(df_agg.index).fillna(0.0).astype(float)

        # build pairs
        pairs = []
        for c100 in [tot100] + cats100:
            if c100 in df_agg.columns:
                c1k = c100.replace("_100m-Gitter", "_1km-Gitter")
                if c1k in df1_sub.columns:
                    pairs.append((c100, c1k))
        if not pairs:
            print(f"- {tot1}: skipped (no comparable columns)")
            continue

        # row-wise vector metrics (categories only)
        cats_pairs = [(c100, c1k) for c100, c1k in pairs if c100 != tot100]
        if not cats_pairs:
            print(f"- {tot1}: no category pairs; skipping row-wise vector metrics")
            continue

        A = np.column_stack([df_agg[c100].to_numpy() for c100, _ in cats_pairs])
        B = np.column_stack([_safe_num(df1_sub[c1k].to_numpy()) for _, c1k in cats_pairs])
        Tparent = _safe_num(df1_sub[tot1].to_numpy()) if tot1 in df1_sub.columns else np.maximum(np.sum(B, axis=1), 1.0)

        row_linf = np.max(np.abs(A - B), axis=1)
        row_l1rel = np.sum(np.abs(A - B), axis=1) / np.maximum(np.abs(Tparent), 1.0)
        flagged = (row_linf > row_linf_thr) | (row_l1rel > row_l1rel_thr)

        # histograms and quantiles for rows
        linf_hist = _hist_counts(row_linf, L_INF_BINS)
        l1r_hist  = _hist_counts(row_l1rel, L1_REL_BINS)

        def _q(x, q):
            return float(np.quantile(x, q)) if x.size else 0.0

        linf_q50, linf_q90, linf_q99 = _q(row_linf, 0.5), _q(row_linf, 0.9), _q(row_linf, 0.99)
        l1r_q50,  l1r_q90,  l1r_q99  = _q(row_l1rel, 0.5), _q(row_l1rel, 0.9), _q(row_l1rel, 0.99)

        # row totals check
        if tot100 in df_agg.columns and tot1 in df1_sub.columns:
            d_tot = df_agg[tot100].to_numpy() - _safe_num(df1_sub[tot1].to_numpy())
            denom = np.maximum(np.abs(_safe_num(df1_sub[tot1].to_numpy())), 1.0)
            ok_tot = (np.abs(d_tot) <= abs_tol) | (np.abs(d_tot) <= rel_tol * denom)
            bad_tot_rows = int((~ok_tot).sum())
            ad = np.abs(d_tot); rd = ad / denom
            max_abs_tot  = float(ad.max()) if ad.size else 0.0
            max_rel_tot  = float(rd.max()) if rd.size else 0.0
            abs_hist_tot = _hist_counts(ad, ABS_BINS_CELL)
            rel_hist_tot = _hist_counts(rd, REL_BINS_CELL)
        else:
            bad_tot_rows = 0
            max_abs_tot = max_rel_tot = 0.0
            abs_hist_tot = [0]*(len(ABS_BINS_CELL)+1)
            rel_hist_tot = [0]*(len(REL_BINS_CELL)+1)

        print(f"- {tot1}: rows={len(row_linf):,} | flagged={int(flagged.sum()):,} ({100*flagged.mean():.3f}%) "
              f"| L∞ max={row_linf.max():.3e} (p50={linf_q50:.3e}, p90={linf_q90:.3e}, p99={linf_q99:.3e}) "
              f"| L1/total max={row_l1rel.max():.3e} (p50={l1r_q50:.3e}, p90={l1r_q90:.3e}, p99={l1r_q99:.3e})")
        print("    " + _fmt_hist(L_INF_BINS, linf_hist, "L∞ bins:"))
        print("    " + _fmt_hist(L1_REL_BINS, l1r_hist,  "L1/total bins:"))

        print(f"  totals check: bad_rows={bad_tot_rows:,} | max abs={max_abs_tot:.3e} | max rel={max_rel_tot:.3e}")
        print("    " + _fmt_hist(ABS_BINS_CELL, abs_hist_tot, "abs bins (totals):"))
        print("    " + _fmt_hist(REL_BINS_CELL, rel_hist_tot, "rel bins (totals):"))

        # global mass (total + per-category)
        if tot100 in df_agg.columns and tot1 in df1_sub.columns:
            mass_child = float(df_agg[tot100].sum())
            mass_par   = float(_safe_num(df1_sub[tot1]).sum())
            diff = mass_child - mass_par
            rel = diff / (mass_par if mass_par != 0 else 1.0)
            print(f"  global mass: TOTAL child={mass_child:.3f} | parent={mass_par:.3f} | diff={diff:.3e} ({rel:.3e} rel)")

        cats_pairs_named = [(c100, c1k, c1k) for (c100, c1k) in cats_pairs]
        cat_deltas = []
        for c100, c1k, name in cats_pairs_named:
            mc = float(df_agg[c100].sum())
            mp = float(_safe_num(df1_sub[c1k]).sum())
            dd = mc - mp
            rr = dd / (mp if mp != 0 else 1.0)
            cat_deltas.append((name, dd, rr))
        if cat_deltas:
            cat_deltas.sort(key=lambda x: abs(x[2]), reverse=True)
            top = cat_deltas[:5]
            wtxt = "; ".join([f"{name}: diff={dd:.3e} ({rr:.3e} rel)" for name, dd, rr in top])
            print(f"  global mass (top-5 cat diffs): {wtxt}")

        # --- diagnostics for why some 1km rows fail (strict) totals check ---
        if tot100 in df_agg.columns and tot1 in df1_sub.columns:
            child_tot = df_agg[tot100].to_numpy()
            parent_tot = _safe_num(df1_sub[tot1].to_numpy())
            d_tot = child_tot - parent_tot

            # buckets
            small_parent = parent_tot <= 10
            zero_child   = child_tot == 0
            zero_parent  = parent_tot == 0
            partial_cov  = (child_tot > 0) & (parent_tot > 0) & (np.abs(d_tot) > ABS_TOL) \
                           & (np.abs(d_tot) > REL_TOL * np.maximum(parent_tot, 1.0))

            print("    diag:")
            print(f"      small parent (≤10): {int(small_parent.sum()):,}")
            print(f"      zero parent: {int(zero_parent.sum()):,} | zero child: {int(zero_child.sum()):,}")
            print(f"      failing w/ partial coverage: {int(partial_cov.sum()):,}")
            if partial_cov.any():
                # show a few worst offenders
                worst_idx = np.argsort(-np.abs(d_tot * partial_cov))[:5]
                for wi in worst_idx:
                    print(f"        • key={df1_sub.index[wi]} parent={parent_tot[wi]:.3f} child={child_tot[wi]:.3f} "
                          f"absdiff={d_tot[wi]:.3f} reldiff={d_tot[wi]/max(parent_tot[wi],1.0):.3e}")

        print("  worst offenders (by |Δ total|):")
        _print_worst_offenders(df_agg, df1_sub, tot100, tot1, cats_pairs, topn=10)


# --- run
sanity_check(PATH_100_FIXED, PATH_1KM,
                              abs_tol=ABS_TOL, rel_tol=REL_TOL,
                              row_linf_thr=ROW_LINF_THR, row_l1rel_thr=ROW_L1REL_THR,
                              batch=BATCH)


Using totals: 100m='_adj', 1km='_adj'; Age POP total='_adj'.


 internal age checks]
rows seen: 3,148,482
POP_TOTAL_100m_adj ≟ ∑AGE_0..100 -> bad rows: 0 (0.0000%) | max abs=1.505e-05 | max rel=3.753e-08
Aggregates (10y & 5cls) vs per-age [target≥50]
  cells=131,381 | mean |diff|=1.5987 | max |diff|=3.879e+01 | mean rel.err=2.3578e-02 | max rel.err=5.541e-01 | bad by tol=51,307
  abs bins: [≤0.25]:18,195 | (0.25,0.5]:16,858 | (0.5,1]:28,707 | (1,2]:35,911 | (2,5]:24,571 | (5,10]:5,714 | >(last):1,425
  rel bins: [≤0.01]:46,666 | (0.01,0.02]:33,408 | (0.02,0.05]:38,711 | (0.05,0.1]:8,444 | (0.1,0.2]:3,614 | (0.2,0.5]:537 | >(last):1
Aggregates (10y & 5cls) vs per-age [target<50]
  cells=22,097,472 | mean |diff|=1.0354 | max |diff|=3.672e+01 | mean rel.err=2.0724e-01 | max rel.err=5.670e+00 | bad by tol=20,254,635
  abs bins: [≤0.25]:3,490,177 | (0.25,0.5]:3,319,966 | (0.5,1]:5,787,879 | (1,2]:7,008,775 | (2,5]:2,427,303 | (5,10]:61,654 | >(last):1,718
  rel bins: [≤0.01]:934,573 | (0.01

In [5]:
# -*- coding: utf-8 -*-
# Reduce file size of the 100m Parquet, and run sanity checks:
# - Remove heavy/redundant cols
# - Downcast floats
# - Run row-level + parent rollup checks while streaming
import re, json, numpy as np, pandas as pd
from collections import defaultdict

import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# ------------------------------------------------------------
# inputs/outputs
# ------------------------------------------------------------
PATH_100M_IN   = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_100m_with_gender_backf_binneds_happyorphans.parquet"  # big 100m file
PATH_1KM_IN    = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_1km_with_binneds.parquet"
PATH_100M_OUT  = r"T:\petre\UCFL\Synthetic Population\Zensus\with_reduced_size\cells_100m_slim.parquet"

BATCH_SIZE = 1_000_000

# ------------------------------------------------------------
# version-agnostic scanner helpers (fix for .scan vs .scanner)
# ------------------------------------------------------------
def make_scanner(dataset, *, columns=None, batch_size=1_000_000, filter=None):
    """
    PyArrow version-agnostic scanner creator.
    - Newer versions: dataset.scan(...)
    - Older versions: dataset.scanner(...)
    """
    if hasattr(dataset, "scan"):  # newer
        return dataset.scan(columns=columns, batch_size=batch_size, filter=filter)
    return dataset.scanner(columns=columns, batch_size=batch_size, filter=filter)

def rb_to_table(rb):
    """Normalize scanner outputs to a pyarrow.Table."""
    if isinstance(rb, pa.RecordBatch):
        return pa.Table.from_batches([rb])
    elif isinstance(rb, pa.Table):
        return rb
    # Some pyarrow versions can yield RecordBatchReader from to_reader()
    if hasattr(rb, "read_all"):  # RecordBatchReader
        return rb.read_all()
    raise TypeError(f"Unsupported batch type: {type(rb)}")

# --- version-tolerant ParquetWriter opener (no WriterProperties) ---
def open_parquet_writer(path, schema):
    # Try most efficient first; progressively remove newer options
    tried = [
        # newer pyarrow builds (zstd + byte_stream_split + compression_level)
        dict(compression="zstd", use_dictionary=True, write_statistics=True,
             use_byte_stream_split=True, compression_level=7),
        # many builds: zstd + byte_stream_split
        dict(compression="zstd", use_dictionary=True, write_statistics=True,
             use_byte_stream_split=True),
        # widely supported across versions
        dict(compression="zstd", use_dictionary=True, write_statistics=True),
        # fallback if zstd codec isn't available in your build
        dict(compression="snappy", use_dictionary=True, write_statistics=True),
        # absolute fallback
        dict(),
    ]
    last_err = None
    for kw in tried:
        try:
            return pq.ParquetWriter(path, schema, **kw)
        except TypeError as e:
            last_err = e
            continue
    # If we get here, something is very old/odd—raise the most recent error
    raise last_err

def iter_record_batches(scanner):
    """
    Yield pyarrow.RecordBatch from a Scanner across versions.
    """
    if hasattr(scanner, "to_reader"):
        yield from scanner.to_reader()
    elif hasattr(scanner, "scan_batches"):
        yield from scanner.scan_batches()
    else:
        tbl = scanner.to_table()
        yield from tbl.to_batches()

# ------------------------------------------------------------
# get topic specs (you already have this in your codebase)
# ------------------------------------------------------------
# must supply: build_topic_specs_for_level("100m")
# Spec fields used:
#   - spec.name
#   - spec.child_cat_cols (100m category columns)
#   - spec.child_row_total_col (100m "Insgesamt_*" total column)
#   - spec.parent_cat_cols (aligned 1km category columns in same order)
topics_100m = build_topic_specs_for_level("100m")  # from your code

# helpers
def is_total_col(c: str) -> bool:
    return c.startswith("Insgesamt_") and c.endswith("_100m-Gitter")

def msg(*a): print(*a, flush=True)

def _norm_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()

# ------------------------------------------------------------
# load schemas / parents
# ------------------------------------------------------------
ds100 = ds.dataset(PATH_100M_IN, format="parquet")
cols100 = set(ds100.schema.names)

df1 = pd.read_parquet(PATH_1KM_IN)  # 1km parent
df1["GITTER_ID_1km"] = _norm_id(df1["GITTER_ID_1km"])

# ------------------------------------------------------------
# 0) Coverage checks (1km parents vs 100m children)
# ------------------------------------------------------------
# Get 1km IDs present in the 100m file by scanning ONLY that column (cheap).
scan_ids = make_scanner(ds100, columns=["GITTER_ID_1km"], batch_size=BATCH_SIZE)
child_1km_ids = set()
for rb in iter_record_batches(scan_ids):
    t = rb_to_table(rb)
    if "GITTER_ID_1km" in t.column_names:
        s = t["GITTER_ID_1km"].to_pandas().astype(str).str.strip()
        child_1km_ids.update(s.unique().tolist())

parent_1km_ids = set(df1["GITTER_ID_1km"].unique())

missing_in_child = parent_1km_ids - child_1km_ids  # 1km parents with NO 100m children
orphans_in_child = child_1km_ids - parent_1km_ids  # 100m rows with UNKNOWN 1km parent

msg("=" * 88)
msg("[COVERAGE] 1km ↔ 100m")
msg("-" * 88)
msg(f"Unique 1km IDs in parent file: {len(parent_1km_ids):,}")
msg(f"Unique 1km IDs referenced by 100m: {len(child_1km_ids):,}")
msg(f"Parents with NO children: {len(missing_in_child):,}")
msg(f"Child groups with UNKNOWN parent: {len(orphans_in_child):,}")
msg("=" * 88)

# ------------------------------------------------------------
# 1) row-level checks at 100 m (per topic + ages), streaming
# ------------------------------------------------------------
row_report = {}
parent_report = {}

# --- AGE checks: build bins from AGE_0..AGE_100 and compare to aggregates (if present)
age_cols = [f"AGE_{i}" for i in range(0, 101) if f"AGE_{i}" in cols100]
have_age = (len(age_cols) == 101)

age10_cols = [
    "Unter10_Alter_in_10er-Jahresgruppen_100m-Gitter",
    "a10bis19_Alter_in_10er-Jahresgruppen_100m-Gitter",
    "a20bis29_Alter_in_10er-Jahresgruppen_100m-Gitter",
    "a30bis39_Alter_in_10er-Jahresgruppen_100m-Gitter",
    "a40bis49_Alter_in_10er-Jahresgruppen_100m-Gitter",
    "a50bis59_Alter_in_10er-Jahresgruppen_100m-Gitter",
    "a60bis69_Alter_in_10er-Jahresgruppen_100m-Gitter",
    "a70bis79_Alter_in_10er-Jahresgruppen_100m-Gitter",
    "a80undaelter_Alter_in_10er-Jahresgruppen_100m-Gitter",
]
age5_cols = [
    "Unter18_Alter_in_5_Altersklassen_100m-Gitter",
    "a18bis29_Alter_in_5_Altersklassen_100m-Gitter",
    "a30bis49_Alter_in_5_Altersklassen_100m-Gitter",
    "a50bis64_Alter_in_5_Altersklassen_100m-Gitter",
    "a65undaelter_Alter_in_5_Altersklassen_100m-Gitter",
]
have_10y = all(c in cols100 for c in age10_cols)
have_5cl = all(c in cols100 for c in age5_cols)

def build_age10_from_micro(A):  # A shape (n,101)
    bins = []
    bins.append(A[:, 0:10].sum(axis=1))      # <10
    for start in range(10, 80, 10):          # 10-19 ... 70-79
        bins.append(A[:, start:start+10].sum(axis=1))
    bins.append(A[:, 80:].sum(axis=1))       # 80+
    return np.vstack(bins).T  # (n,9)

def build_age5_from_micro(A):
    return np.vstack([
        A[:, :18].sum(axis=1),   # <18
        A[:, 18:30].sum(axis=1), # 18-29
        A[:, 30:50].sum(axis=1), # 30-49
        A[:, 50:65].sum(axis=1), # 50-64
        A[:, 65:].sum(axis=1),   # 65+
    ]).T  # (n,5)

age_row_counts = defaultdict(int)
age_row_maxerr = defaultdict(float)
age_row_meanabs = defaultdict(float)  # running mean via sums
sum_abs_err_10 = 0.0
sum_abs_err_5  = 0.0

if have_age and (have_10y or have_5cl):
    need_cols = ["GITTER_ID_1km"] + age_cols
    if have_10y: need_cols += age10_cols
    if have_5cl: need_cols += age5_cols

    scanner = make_scanner(ds100, columns=need_cols, batch_size=BATCH_SIZE)
    for rb in iter_record_batches(scanner):
        t = rb_to_table(rb)
        # we don't actually need pandas dtypes here; Arrow->np is fine
        dfb = t.to_pandas()

        A = dfb[age_cols].to_numpy(dtype=float)  # (m,101)

        if have_10y:
            built9 = build_age10_from_micro(A)
            given9 = dfb[age10_cols].to_numpy(dtype=float)
            err = np.abs(built9 - given9)
            if err.size:
                age_row_maxerr["10y"] = max(age_row_maxerr["10y"], float(np.nanmax(err)))
                sum_abs_err_10 += float(np.nansum(err))
            age_row_counts["10y"] += len(dfb)

        if have_5cl:
            built5 = build_age5_from_micro(A)
            given5 = dfb[age5_cols].to_numpy(dtype=float)
            err = np.abs(built5 - given5)
            if err.size:
                age_row_maxerr["5cl"] = max(age_row_maxerr["5cl"], float(np.nanmax(err)))
                sum_abs_err_5 += float(np.nansum(err))
            age_row_counts["5cl"] += len(dfb)

    if have_10y:
        age_row_meanabs["10y"] = sum_abs_err_10 / max(age_row_counts["10y"] * 9, 1)
    if have_5cl:
        age_row_meanabs["5cl"] = sum_abs_err_5 / max(age_row_counts["5cl"] * 5, 1)

# --- Per-topic row consistency & 100m→1km rollup vs parent
def topic_row_and_parent_checks(spec):
    name = spec.name
    cats = [c for c in spec.child_cat_cols if c in cols100]
    total = spec.child_row_total_col if spec.child_row_total_col in cols100 else None

    if not cats:
        row_report[name] = {"status": "skip (no child cols at 100m)"}
        parent_report[name] = {"status": "skip"}
        return

    # row-level counters
    cnt = 0
    abs_gt_1e2 = 0
    abs_gt_1e1 = 0
    abs_gt_1e2m = 0  # > 1
    max_abs = 0.0
    sum_abs = 0.0

    # parent rollup accumulator: 1km -> per-category sums
    rollups = defaultdict(lambda: defaultdict(float))  # {cat: {gid1: sum}}

    need_cols = ["GITTER_ID_1km"] + cats + ([total] if total else [])
    scanner = make_scanner(ds100, columns=need_cols, batch_size=BATCH_SIZE)

    for rb in iter_record_batches(scanner):
        t = rb_to_table(rb)
        dfb = t.to_pandas()

        # row check
        if total:
            s = dfb[cats].sum(axis=1).to_numpy(float)
            g = pd.to_numeric(dfb[total], errors="coerce").fillna(0.0).to_numpy(float)
            diff = s - g
            ad = np.abs(diff)
            if ad.size:
                max_abs = max(max_abs, float(np.max(ad)))
                sum_abs += float(np.sum(ad))
                abs_gt_1e2  += int(np.sum(ad > 1e-2))
                abs_gt_1e1  += int(np.sum(ad > 1e-1))
                abs_gt_1e2m += int(np.sum(ad > 1))
                cnt += len(dfb)

        # parent rollup
        if len(dfb):
            grp = dfb.groupby("GITTER_ID_1km", sort=False, observed=True)
            sums = grp[cats].sum(numeric_only=True)
            for cat in cats:
                v = sums[cat].to_dict()
                for gid1, val in v.items():
                    rollups[cat][gid1] += float(val)

    # write row report
    if total:
        row_report[name] = {
            "rows_checked": cnt,
            "abs>1e-2": abs_gt_1e2,
            "abs>1e-1": abs_gt_1e1,
            "abs>1": abs_gt_1e2m,
            "max_abs": max_abs,
            "mean_abs": (sum_abs / max(cnt, 1)),
            "total_col": total,
        }
    else:
        row_report[name] = {"status": "no total col present at 100m"}

    # parent compare: merge with 1km parent category cols
    parent_cols = [c for c in spec.parent_cat_cols if c in df1.columns]
    if not parent_cols:
        parent_report[name] = {"status": "skip (no parent cols at 1km)"}
        return

    # assemble gid1 x cats from rollups
    gids = set()
    for cat in cats:
        gids.update(rollups[cat].keys())
    gid_list = pd.Index(sorted(gids))
    roll_df = pd.DataFrame(index=gid_list)
    for cat in cats:
        s = pd.Series(rollups[cat], dtype="float64")
        roll_df[cat] = s.reindex(gid_list).fillna(0.0)

    # parent slice: ensure IDs align
    p = df1.set_index("GITTER_ID_1km")
    parent_map = dict(zip(spec.child_cat_cols, spec.parent_cat_cols))
    parent_use = [parent_map[c] for c in cats if c in parent_map and parent_map[c] in p.columns]
    parent_df = p.reindex(gid_list)[parent_use].fillna(0.0)

    # rename parent->child for aligned diff reporting
    parent_df.columns = cats[:len(parent_df.columns)]
    diff = roll_df[cats[:len(parent_df.columns)]] - parent_df

    # summarize per-category
    stats = {}
    for col in diff.columns:
        d = diff[col].to_numpy()
        stats[col] = {
            "max_abs": float(np.max(np.abs(d))) if d.size else 0.0,
            "mean_abs": float(np.mean(np.abs(d))) if d.size else 0.0,
            "abs>1e-2": int(np.sum(np.abs(d) > 1e-2)),
            "abs>1e-1": int(np.sum(np.abs(d) > 1e-1)),
            "abs>1":    int(np.sum(np.abs(d) > 1.0)),
            "n_1km":    int(diff.shape[0]),
        }

    parent_report[name] = {
        "n_1km_rows": int(diff.shape[0]),
        "per_category": stats,
    }

# run topic checks
for spec in topics_100m:
    msg(f"checking topic: {spec.name}")
    topic_row_and_parent_checks(spec)

# ------------------------------------------------------------
# 2) print/report diagnostics
# ------------------------------------------------------------
msg("\n=== AGE checks (rebuilt from AGE_0..100) ===")
if have_age:
    if have_10y:
        msg(f"10-year bins: rows ≈ {age_row_counts['10y']:,} | mean |err| per cell = {age_row_meanabs['10y']:.6g} | max |err| = {age_row_maxerr['10y']:.6g}")
    else:
        msg("10-year bins: not present at 100m")
    if have_5cl:
        msg(f"5-class bins: rows ≈ {age_row_counts['5cl']:,} | mean |err| per cell = {age_row_meanabs['5cl']:.6g} | max |err| = {age_row_maxerr['5cl']:.6g}")
    else:
        msg("5-class bins: not present at 100m")
else:
    msg("AGE_0..AGE_100 not present; skipping age plausibility.")

msg("\n=== Row-wise totals vs sum(categories) (100m) ===")
for k, v in row_report.items():
    msg(k, "->", json.dumps(v, ensure_ascii=False))

msg("\n=== 100m→1km parent consistency (per-category) ===")
for k, v in parent_report.items():
    if "status" in v:
        msg(k, "->", v["status"])
        continue
    msg(k, "->", f"n_1km={v['n_1km_rows']:,}")
    if "per_category" in v:
        bad_1pct = sum(1 for s in v["per_category"].values() if s["abs>1e-2"] > 0)
        msg(f"   categories with any |diff|>1e-2: {bad_1pct}/{len(v['per_category'])}")

# ------------------------------------------------------------
# 3) write slim 100m parquet
#     (keep POP_TOTAL_100m_adj; drop POP_TOTAL_100m and all totals/age-aggregates)
# ------------------------------------------------------------
msg("\n=== Writing slim 100m parquet ===")

# columns to keep (strict minimal)
KEEP = {"GITTER_ID_100m", "GITTER_ID_1km", "POP_TOTAL_100m_adj"}
for spec in topics_100m:
    for c in spec.child_cat_cols:
        if c in cols100:
            KEEP.add(c)
# keep micro ages if present
for c in age_cols:
    KEEP.add(c)

# drop patterns (we already go "strict minimal" below, but keep for clarity)
DROP_PATTERNS = [
    r"^werterlaeuternde_Zeichen_",
    r"^scale$", r"^is_orphan$",
    r"^POP_TOTAL_100m$",  # drop non-adj
    # drop age aggregates
    r"_Alter_in_10er-Jahresgruppen_100m-Gitter$",
    r"_Alter_in_5_Altersklassen_100m-Gitter$",
]
def should_drop(c: str) -> bool:
    if is_total_col(c):             return True
    for p in DROP_PATTERNS:
        if re.search(p, c):         return True
    return False

# strict minimal file: only KEEP
final_cols = sorted(list(KEEP))

scanner = make_scanner(ds100, columns=final_cols, batch_size=BATCH_SIZE)

writer = None
for rb in iter_record_batches(scanner):
    t = rb_to_table(rb)  # (use the helper from earlier reply)
    # cast float64 -> float32 to shrink
    arrays, fields = [], []
    for f, arr in zip(t.schema, t.columns):
        if pa.types.is_floating(f.type) and f.type.bit_width == 64:
            arr = pa.compute.cast(arr, pa.float32())
            fields.append(pa.field(f.name, pa.float32()))
        else:
            fields.append(pa.field(f.name, f.type))
        arrays.append(arr)
    t32 = pa.Table.from_arrays(arrays, names=[f.name for f in fields])

    if writer is None:
        writer = open_parquet_writer(PATH_100M_OUT, t32.schema)
    writer.write_table(t32)

if writer:
    writer.close()

msg("done.")


[COVERAGE] 1km ↔ 100m
----------------------------------------------------------------------------------------
Unique 1km IDs in parent file: 212,758
Unique 1km IDs referenced by 100m: 211,486
Parents with NO children: 1,314
Child groups with UNKNOWN parent: 42
checking topic: Familienstand
checking topic: Energietraeger
checking topic: Heizungsart
checking topic: Haushaltsgroesse
checking topic: Lebensform
checking topic: Raeume
checking topic: Wohnflaeche
checking topic: Geburtsland

=== AGE checks (rebuilt from AGE_0..100) ===
10-year bins: rows ≈ 3,148,482 | mean |err| per cell = 0.824418 | max |err| = 38.7871
5-class bins: rows ≈ 3,148,482 | mean |err| per cell = 0.992539 | max |err| = 38.6711

=== Row-wise totals vs sum(categories) (100m) ===
Familienstand -> {"rows_checked": 3148482, "abs>1e-2": 2844513, "abs>1e-1": 1812227, "abs>1": 141142, "max_abs": 12.101941394386813, "mean_abs": 0.2626498462809062, "total_col": "Insgesamt_Bevoelkerung_Familienstand_100m-Gitter"}
Energietrae

In [6]:
import zlib, math
import numpy as np
import pyarrow as pa
import pyarrow.dataset as ds

# --- reuse your helpers if you already added them ---
def make_scanner(dataset, *, columns=None, batch_size=1_000_000, filter=None):
    if hasattr(dataset, "scan"):
        return dataset.scan(columns=columns, batch_size=batch_size, filter=filter)
    return dataset.scanner(columns=columns, batch_size=batch_size, filter=filter)

def iter_record_batches(scanner):
    if hasattr(scanner, "to_reader"):
        yield from scanner.to_reader()
    elif hasattr(scanner, "scan_batches"):
        yield from scanner.scan_batches()
    else:
        tbl = scanner.to_table()
        yield from tbl.to_batches()

def rb_to_table(rb):
    if isinstance(rb, pa.RecordBatch):
        return pa.Table.from_batches([rb])
    if hasattr(rb, "read_all"):
        return rb.read_all()
    if isinstance(rb, pa.Table):
        return rb
    raise TypeError(type(rb))

# --- files ---
PATH_BIG  = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_100m_with_gender_backf_binneds_happyorphans.parquet"
PATH_SLIM = r"T:\petre\UCFL\Synthetic Population\Zensus\with_reduced_size\cells_100m_slim.parquet"

ds_big  = ds.dataset(PATH_BIG,  format="parquet")
ds_slim = ds.dataset(PATH_SLIM, format="parquet")

# columns we expect in slim (just read from slim)
slim_cols = ds_slim.schema.names
id_col    = "GITTER_ID_100m"
assert id_col in slim_cols, f"{id_col} missing in slim"

# Which numeric columns to compare (skip obvious non-numerics)
num_cols = []
for f in ds_slim.schema:
    if pa.types.is_integer(f.type) or pa.types.is_floating(f.type):
        num_cols.append(f.name)

# ---------- helpers ----------
def count_rows(dataset, *, batch_size=1_000_000):
    n = 0
    sc = make_scanner(dataset, columns=[id_col], batch_size=batch_size)
    for rb in iter_record_batches(sc):
        t = rb_to_table(rb)
        n += t.num_rows
    return n

def adler32_checksum(dataset, *, col=id_col, batch_size=1_000_000):
    chk = 1  # adler32 initial value
    sc = make_scanner(dataset, columns=[col], batch_size=batch_size)
    for rb in iter_record_batches(sc):
        t = rb_to_table(rb)
        # strings can be large; convert batch-wise
        arr = t.column(col).cast(pa.string())
        # join with a separator to avoid accidental collisions for ["ab","c"] vs ["a","bc"]
        payload = ("\n".join([x.as_py() or "" for x in arr])).encode("utf-8", errors="ignore")
        chk = zlib.adler32(payload, chk)
    return chk & 0xFFFFFFFF

def numeric_aggregates(dataset, cols, *, batch_size=500_000):
    stats = {c: {"sum":0.0, "min": math.inf, "max": -math.inf, "nnz":0, "nulls":0} for c in cols}
    sc = make_scanner(dataset, columns=cols, batch_size=batch_size)
    for rb in iter_record_batches(sc):
        t = rb_to_table(rb)
        for c in cols:
            col = t[c]
            # null handling
            nulls = col.null_count
            n = len(col) - nulls
            stats[c]["nulls"] += int(nulls)
            stats[c]["nnz"]   += int(n)
            # min/max need care when there are nulls
            # use pyarrow compute for robust min/max (fast C++)
            if n:
                mn = pa.compute.min(col).as_py()
                mx = pa.compute.max(col).as_py()
                stats[c]["min"] = mn if stats[c]["min"] is math.inf else min(stats[c]["min"], mn)
                stats[c]["max"] = mx if stats[c]["max"] is -math.inf else max(stats[c]["max"], mx)
                # sum as float64 for stability
                s = pa.compute.sum(col.cast(pa.float64())).as_py() or 0.0
                stats[c]["sum"] += float(s)
    # post-fix infinities if column was all-null
    for c, d in stats.items():
        if d["nnz"] == 0:
            d["min"] = None
            d["max"] = None
    return stats

def pretty_diff(a, b, atol=1e-4, rtol=1e-6):
    # returns (ok, abs_diff, rel_diff)
    ad = abs(a - b)
    rd = ad / max(abs(b), 1e-12)
    ok = (ad <= atol) or (rd <= rtol)
    return ok, ad, rd

# ---------- run smoke checks ----------
print("=== ROW COUNT ===")
n_big  = count_rows(ds_big)
n_slim = count_rows(ds_slim)
print(f"big :  {n_big:,}")
print(f"slim:  {n_slim:,}")
print("OK" if n_big == n_slim else "MISMATCH")

print("\n=== ID CHECKSUM (Adler32 over GITTER_ID_100m) ===")
chk_big  = adler32_checksum(ds_big)
chk_slim = adler32_checksum(ds_slim)
print(f"big : 0x{chk_big:08x}")
print(f"slim: 0x{chk_slim:08x}")
print("OK (same ID multiset)" if chk_big == chk_slim else "⚠️ Different ID sequence/multiset")

print("\n=== SCHEMA & DTYPES (slim) ===")
print("columns:", slim_cols)
float32_cols = [f.name for f in ds_slim.schema if pa.types.is_floating(f.type) and f.type.bit_width == 32]
print("float32 columns (sample):", float32_cols[:10], " … total:", len(float32_cols))

print("\n=== NUMERIC AGGREGATES (sum/min/max/nulls) ===")
agg_big  = numeric_aggregates(ds_big,  num_cols)
agg_slim = numeric_aggregates(ds_slim, num_cols)

# compare with loose tolerances due to float32 cast
ATOL = 1e-3   # absolute tolerance
RTOL = 1e-6   # relative tolerance

bad = []
for c in num_cols:
    a, b = agg_slim[c], agg_big[c]
    ok_sum, ad_sum, rd_sum = pretty_diff(a["sum"], b["sum"], atol=ATOL, rtol=RTOL)
    ok_min = (a["min"] == b["min"]) or (a["min"] is None and b["min"] is None) or (a["min"] is not None and b["min"] is not None and abs(a["min"]-b["min"]) <= ATOL)
    ok_max = (a["max"] == b["max"]) or (a["max"] is None and b["max"] is None) or (a["max"] is not None and b["max"] is not None and abs(a["max"]-b["max"]) <= ATOL)
    ok_nulls = (a["nulls"] == b["nulls"]) and (a["nnz"] == b["nnz"])
    if not (ok_sum and ok_min and ok_max and ok_nulls):
        bad.append((c, {"sum_abs": ad_sum, "sum_rel": rd_sum, "min_ok": ok_min, "max_ok": ok_max, "nulls_ok": ok_nulls}))

    # print a one-line summary per column (short)
    print(f"{c:>35s} | sumΔ_abs={ad_sum:.3g} rel={rd_sum:.3g} | min:{'ok' if ok_min else 'bad'} max:{'ok' if ok_max else 'bad'} nulls:{'ok' if ok_nulls else 'bad'}")

if bad:
    print("\nColumns with notable differences beyond tolerance (float32 cast can cause tiny Δ):")
    for c, d in bad[:15]:
        print(f" - {c}: {d}")
else:
    print("\nAll checked numeric columns within tolerances")


=== ROW COUNT ===
big :  3,148,482
slim:  3,148,482
OK

=== ID CHECKSUM (Adler32 over GITTER_ID_100m) ===
big : 0x1a86c8cc
slim: 0x1a86c8cc
OK (same ID multiset)

=== SCHEMA & DTYPES (slim) ===
columns: ['100bis109_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '110bis119_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '120bis129_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '130bis139_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '140bis149_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '150bis159_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '160bis169_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '170bis179_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '180undmehr_Flaeche_der_Wohnung_10m2_Intervalle_100m-Gitter', '1Raum_Wohnungen_nach_Zahl_der_Raeume_100m-Gitter', '1_Person_Groesse_des_privaten_Haushalts_100m-Gitter', '2Raeume_Wohnungen_nach_Zahl_der_Raeume_100m-Gitter', '2_Personen_Groesse_des_privaten_Haushalts_100m-Gitter', '30bis39_Flaeche_der_Wohnung

In [8]:
import pandas as pd
from pathlib import Path

PATH_100M_OUT = Path(r"T:\petre\UCFL\Synthetic Population\Zensus\with_reduced_size\cells_100m_slim.parquet")
PATH_100M_CSV = PATH_100M_OUT.with_suffix(".csv")

# Load fully (fast path via pyarrow)
df = pd.read_parquet(PATH_100M_OUT, engine="pyarrow")

# Quick peek + memory footprint
print(df.shape, "rows x cols")
print("RAM ~", round(df.memory_usage(deep=True).sum()/1e9, 3), "GB")

# Write CSV (single shot)
df.to_csv(PATH_100M_CSV, index=False)
print("Wrote:", PATH_100M_CSV)


(3148482, 170) rows x cols
RAM ~ 2.616 GB
Wrote: T:\petre\UCFL\Synthetic Population\Zensus\with_reduced_size\cells_100m_slim.csv


In [9]:
# For debug

import pyarrow.dataset as ds
import re

PATH_100 = r"T:\petre\UCFL\Synthetic Population\Zensus\with_other_binned_data\cells_100m_with_gender_backf_binneds.parquet"

ds100 = ds.dataset(PATH_100, format="parquet")
cols  = ds100.schema.names

# adj_cols = [c for c in cols if c.startswith("Insgesamt")]
# print(f"adj columns found: {len(adj_cols)}")
# for c in sorted(adj_cols)[:50]:
#     print("  ", c)
# if len(adj_cols) > 50:
#     print("  ...")

for c in cols[:500]:
    print("  ", c)
if len(cols) > 500:
    print("  ...")


   GITTER_ID_100m
   Insgesamt_Bevoelkerung_Alter_in_10er-Jahresgruppen_100m-Gitter
   Unter10_Alter_in_10er-Jahresgruppen_100m-Gitter
   a10bis19_Alter_in_10er-Jahresgruppen_100m-Gitter
   a20bis29_Alter_in_10er-Jahresgruppen_100m-Gitter
   a30bis39_Alter_in_10er-Jahresgruppen_100m-Gitter
   a40bis49_Alter_in_10er-Jahresgruppen_100m-Gitter
   a50bis59_Alter_in_10er-Jahresgruppen_100m-Gitter
   a60bis69_Alter_in_10er-Jahresgruppen_100m-Gitter
   a70bis79_Alter_in_10er-Jahresgruppen_100m-Gitter
   a80undaelter_Alter_in_10er-Jahresgruppen_100m-Gitter
   Insgesamt_Bevoelkerung_Alter_in_5_Altersklassen_100m-Gitter
   Unter18_Alter_in_5_Altersklassen_100m-Gitter
   a18bis29_Alter_in_5_Altersklassen_100m-Gitter
   a30bis49_Alter_in_5_Altersklassen_100m-Gitter
   a50bis64_Alter_in_5_Altersklassen_100m-Gitter
   a65undaelter_Alter_in_5_Altersklassen_100m-Gitter
   AnteilAuslaender_Anteil_Auslaender_100m-Gitter
   AnteilUeber65_Anteil_ueber_65_100m-Gitter
   AnteilUnter18_Anteil_unter_18_100m-G